# Investigating Length Issues with pseudo-documents

>Ziqian Peng, Rachel Bawden, and François Yvon. 2025. Investigating Length Issues in Document-level Machine Translation. In Proceedings of Machine Translation Summit XX: Volume 1, pages 4–23, Geneva, Switzerland. European Association for Machine Translation.

In [34]:
import pandas as pd
from eval_windows import BleuEvaluator
from eval_windows import CometEvaluator
from eval_position_bias import PositionBiasAnalyzer

rename_dict = {
    'Llama-3.1-8B-Instruct': 'Llama',
    'EuroLLM-22B-Instruct'	: 'EuroLLM22',
    # 'Qwen35-0.8B': 'Qwen35-0.8B',
    'Qwen35-9B': 'Qwen35-9B',
}

format_rename_dict = {
    'EuroLLM22': r'EuroLLM$_{22}$',
    'Qwen35-9B': r'Qwen{3.5}$_{9\text{B}}$',
}


model_list = list(rename_dict.keys())

sys_fname_dict = {rename_dict[m]: m for m in model_list }


In [35]:

##########################
# BLEU 
###########################

def compute_bleu_scores(bleu_evaluator: BleuEvaluator, sys_fname_dict, model_list, to_realign = True):
    print('Compute BLEU')
    res_dict = {}
    for model_type in model_list:  
        print(model_type)
        res_dict[model_type] = bleu_evaluator.compute_bleu( model_type, to_realign = to_realign)
        
    # ds-BLEU
    print('Compute ds-BLEU')
    bleu_evaluator.eval_ds_bleu_corpus(sys_fname_dict)


def show_bleu_scores(bleu_evaluator: BleuEvaluator, sys_fname_dict, max_n_order):
    print('BLEU')
    bleu_dict = bleu_evaluator.get_bleu_corpus_dict(sys_fname_dict)
    
    for subset in bleu_evaluator.get_sub_corpus_list():
        print(subset)
        bleu_df = pd.DataFrame(bleu_dict[subset]).loc[max_n_order]
        display(bleu_df)
        print(bleu_df.style.to_latex())
    
    print('ds-BLEU')
    res_dict = bleu_evaluator.get_ds_bleu_corpus_dict(sys_fname_dict, small_bp_size = True)
    for subset in bleu_evaluator.get_sub_corpus_list():
        print(subset)
        
        ds_bleu_df = pd.DataFrame(res_dict[subset]).loc[max_n_order]
        display(ds_bleu_df)
        print(ds_bleu_df.style.to_latex())

    
def show_ttest_ds_bleu(bleu_evaluator: BleuEvaluator, to_test_max_n, sys_fname_dict ):

    ds_bleu_dict = bleu_evaluator.get_ds_bleu_dict(sys_fname_dict)
    
    res_dict = {}
    for model_type in sys_fname_dict.keys():  
        print(model_type)
        res_dict[model_type] = bleu_evaluator.get_test( ds_bleu_dict[model_type], to_test_max_n, verbose = False )
    
    res_df = pd.DataFrame(res_dict)
    display(res_df)
    print(res_df.style.to_latex())
    
    # show p-value
    res_dict = {}
    for model_type in sys_fname_dict.keys():  
        print(model_type)
        res_dict[model_type] = bleu_evaluator.get_test( ds_bleu_dict[model_type], to_test_max_n, verbose = True )
    display(pd.DataFrame(res_dict))

##########################
# COMET
###########################

def show_ttest_comet(comet_evaluator: CometEvaluator, to_test_max_n, sys_fname_dict ):
    
    ds_comet_dict = comet_evaluator.get_ds_comet_dict(sys_fname_dict)
    
    res_dict = {}
    for model_type in sys_fname_dict.keys():  
        print(model_type)
        res_dict[model_type] = comet_evaluator.get_test_comet(  ds_comet_dict[model_type], to_test_max_n, show_p_val = False )

    res_df = pd.DataFrame(res_dict)
    display(res_df)
    print(res_df.style.to_latex())
    
    ##########
    res_dict = {}
    for model_type in sys_fname_dict.keys():  
        print(model_type)
        res_dict[model_type] = comet_evaluator.get_test_comet(  ds_comet_dict[model_type], to_test_max_n, show_p_val = True )
    display(pd.DataFrame(res_dict))



In [36]:

##########################
# Position bias
###########################

def get_nb_sample_for_position_bias(pos_analyser:PositionBiasAnalyzer, sys_fname_dict, nb_positions, disable_full_doc = True):     
    # count the number of sentences translated in m (the max occurances without counting sent-level corpus) different begin idx
    print("set of different begin idx")
    
    test_nb_total = []
    for tokenizer_name in sys_fname_dict.keys():
        total = pos_analyser.count_begin_id_stat_total( tokenizer_name, nb_positions, disable_full_doc = disable_full_doc ) 
        test_nb_total.append(total)
    
    
    assert len(set(test_nb_total)) == 1
    total = test_nb_total[0]
    return total

def check_sets(pos_analyser:PositionBiasAnalyzer, sys_fname_dict, nb_positions, disable_full_doc = True):     
    set_list = []
    for tokenizer_name in sys_fname_dict.keys():
        sent_id_dict = pos_analyser.extract_begin_id_stat(  tokenizer_name, nb_positions, disable_full_doc = disable_full_doc ) 
        set_list.append(sent_id_dict)
    
    # the set of sentences identified by each tokenizer should be identical
    res = all(d == set_list[0] for d in set_list[1:])
    return res

def compute_position_bias(
    pos_analyser:PositionBiasAnalyzer, 
    sys_fname_dict, 
    nb_positions, 
    disable_full_doc = True, 
    show_p_val = True,
):
    assert check_sets(pos_analyser, sys_fname_dict, nb_positions, disable_full_doc)

    res_list = []
    for tokenizer_name, md_fname in sys_fname_dict.items():
        sent_id_dict = pos_analyser.extract_begin_id_stat(  tokenizer_name, nb_positions, disable_full_doc = disable_full_doc ) 
    
        begin_id_stat_dict = pos_analyser.extract_begin_id_stat_testsets(  sent_id_dict, tokenizer_name, disable_full_doc = disable_full_doc ) 
        print(sum(len(v) for v in sent_id_dict.values()))
        print(sum(len(v) for v in begin_id_stat_dict.values()))
    
        tmp_dict = {tokenizer_name: md_fname}
        tmp_df = pos_analyser.get_ttest_testsets(tmp_dict,  begin_id_stat_dict,  nb_positions, show_p_val = show_p_val)
        
        print(tokenizer_name)
        
        tmp_df = tmp_df.reset_index()
        tmp_df.index =  [f"$p${i}-$p${i+1}" for i in range(len(tmp_df.index))]
        res_list.append(tmp_df)

    res_df = pd.concat(res_list, axis = 1)
    display(res_df)
    print(res_df.style.to_latex())
    display(res_df[sys_fname_dict.keys()])
    print(res_df[sys_fname_dict.keys()].style.to_latex())
    


In [37]:
from collections import Counter
from data_paths import SystemPaths, DatasetPaths

def find_repeated_ngrams(text, n):
    # Tokenize the text
    words = text.lower().split()
    
    # Generate n-grams
    ngrams = [tuple(words[i:i+n]) for i in range(len(words)-n+1)]
    
    # Count n-gram frequencies
    ngram_counts = Counter(ngrams)
    
    # Filter out n-grams that appear only once
    repeated_ngrams = {ngram: count for ngram, count in ngram_counts.items() if count > 1 }
    
    return repeated_ngrams

def find_repeated_ngram_doc(text, n, verbose = False, min_repeat = 1 ):
    words = text.lower().split()
    # dict of repeated ngram : count
    repeated_ngrams = find_repeated_ngrams(text, n)

    find = False
    find_id = len(text.split())
    i = 0
    while not find:
        if i+n >= len(words)-n+1 :
            break
        if ' '.join(words[i:i+n]) in [' '.join(k) for k in repeated_ngrams.keys() if repeated_ngrams[k] > min_repeat]:
            # find the index from which the first repetition begin
            # print(' '.join(words[i:i+n]))
            find = True
            find_id = i+n #len(' '.join(words[0:i+n]) )
            if verbose:
                print(f"{n}gram repetition begin at {find_id}, ngram =\n {' '.join(words[i:i+n])}")
        i = i+1
    return len(words), find_id


def find_repeated_ngram_testset(text_path, n, verbose = False, min_repeat = 1 ):
    text_ls = open(text_path, 'r').read().strip().split('\n')
    res_dict = {'nb_words': [], 'repeat_begin': []}
    for text in text_ls:
        nb_words, repeat_begin = find_repeated_ngram_doc(text, n, verbose, min_repeat )
        res_dict['nb_words'] += [ nb_words]
        res_dict['repeat_begin'] += [ repeat_begin]
    return res_dict


    
def get_repeat_stat(
    sys_fname_dict, 
    dataset_paths: DatasetPaths, 
    system_paths: SystemPaths, 
    sub_corpus_list, 
    n = 10,
    verbose = False,
    min_repeat = 1, 
):
    # depend on the function `find_repeated_ngram_testset`
    print(f"ratio of document containing {n}gram")
    res_dict = {}
    for max_n in list(dataset_paths.get_valid_max_n()) + ['doc']:
        res_dict[max_n]= {}

        for md in ['ref'] + list(sys_fname_dict.keys()):
            # print(max_n, md)

            res_dict[max_n][md] = {}
            nb_doc = 0
            nb_repeat = 0
            for subset in sub_corpus_list:

                if md == 'ref':
                    text_path = dataset_paths.testset_ref_path( max_n, subset)
                else:
                    md_fname = sys_fname_dict[md]
                    text_path = system_paths.sys_path(md_fname, max_n, subset)
                
                tmp_res_dict = find_repeated_ngram_testset(text_path, n, verbose = verbose, min_repeat = min_repeat )
                
                tmp_res_df = pd.DataFrame(tmp_res_dict)
                # display(the_res_df)
                
                diff_df =tmp_res_df['nb_words'] - tmp_res_df['repeat_begin']
                

                nb_doc += len(tmp_res_df)
                nb_repeat += len(diff_df[diff_df==0])
                
            res_dict[max_n][md] = nb_repeat/nb_doc
    
    res_df = 1 - pd.DataFrame(res_dict)
    
    # res_df = res_df.rename(index = rename_dict)
    # print(res_df.fillna('-').style.format(precision = 2).to_latex())
    return res_df
    
    

# TEDWidow

In [38]:
from data_paths import (
    TEDPaths,
    TEDSystemPaths,
)


valid_max_n =  {256, 512, 768, 1024, 1200, 1600, 2048}
max_n_order = ['sent'] + sorted(list(valid_max_n)) + ['doc']

corpus_name = 'TED'
sub_corpus_list = ['2014', '2015', '2016', '2017']

DATA_DIR = "/Users/ziqianpeng/MaTOS/data/TAL/corpora/testsets"
ted_data_path = TEDPaths(root = f'{DATA_DIR}/TED')

SYS_DIR = "/Users/ziqianpeng/MaTOS/analyse_result/data_res/TED/eval_llm_output-window_update"
ted_mt_path = TEDSystemPaths(root = f'{SYS_DIR}/TED')


Supported max_n values: {256, 512, 1600, 768, 1024, 2048, 'doc', 'sent', 1200}
Supported max_n values: {256, 512, 1600, 768, 1024, 2048, 'doc', 'sent', 1200}


In [27]:
ngram_df = get_repeat_stat(
    sys_fname_dict, 
    ted_data_path,
    ted_mt_path, 
    sub_corpus_list,
    n = 10,
    verbose = False,
    min_repeat = 1, 
)

display(ngram_df.round(2))
print(ngram_df.style.format(precision = 2).to_latex(column_format="l" + "r" * len(ngram_df.columns)))

ratio of document containing 10gram


,256,512,1600,768,1024,2048,1200,doc
ref,0.01,0.02,0.08,0.03,0.06,0.10,0.07,0.19
Llama,0.01,0.04,0.15,0.06,0.11,0.21,0.12,0.37
EuroLLM22,0.02,0.03,0.13,0.05,0.07,0.16,0.11,0.35
Qwen35-9B,0.01,0.02,0.09,0.04,0.06,0.12,0.07,0.29


\begin{tabular}{lrrrrrrrr}
 & 256 & 512 & 1600 & 768 & 1024 & 2048 & 1200 & doc \\
ref & 0.01 & 0.02 & 0.08 & 0.03 & 0.06 & 0.10 & 0.07 & 0.19 \\
Llama & 0.01 & 0.04 & 0.15 & 0.06 & 0.11 & 0.21 & 0.12 & 0.37 \\
EuroLLM22 & 0.02 & 0.03 & 0.13 & 0.05 & 0.07 & 0.16 & 0.11 & 0.35 \\
Qwen35-9B & 0.01 & 0.02 & 0.09 & 0.04 & 0.06 & 0.12 & 0.07 & 0.29 \\
\end{tabular}



In [41]:
ngram_df = get_repeat_stat(
    sys_fname_dict, 
    ted_data_path,
    ted_mt_path, 
    sub_corpus_list,
    n = 10,
    verbose = False,
    min_repeat = 2, 
)

display(ngram_df.round(3))
print(ngram_df.style.format(precision = 2).to_latex(column_format="l" + "r" * len(ngram_df.columns)))

,256,512,1600,768,1024,2048,1200,doc
ref,0.000,0.000,0.00,0.000,0.000,0.000,0.000,0.000
Llama,0.002,0.004,0.01,0.005,0.014,0.012,0.008,0.019
EuroLLM22,0.002,0.004,0.01,0.005,0.007,0.012,0.008,0.038
Qwen35-9B,0.000,0.000,0.01,0.005,0.007,0.012,0.008,0.038


\begin{tabular}{lrrrrrrrr}
 & 256 & 512 & 1600 & 768 & 1024 & 2048 & 1200 & doc \\
ref & 0.00 & 0.00 & 0.00 & 0.00 & 0.00 & 0.00 & 0.00 & 0.00 \\
Llama & 0.00 & 0.00 & 0.01 & 0.01 & 0.01 & 0.01 & 0.01 & 0.02 \\
EuroLLM22 & 0.00 & 0.00 & 0.01 & 0.01 & 0.01 & 0.01 & 0.01 & 0.04 \\
Qwen35-9B & 0.00 & 0.00 & 0.01 & 0.01 & 0.01 & 0.01 & 0.01 & 0.04 \\
\end{tabular}



### 1. BLEU

In [5]:
ted_bleu = BleuEvaluator(
    dataset_paths = ted_data_path,
    system_paths = ted_mt_path,
    valid_max_n = valid_max_n,
    corpus_name = corpus_name,
    sub_corpus_list = sub_corpus_list,
    sep_tag = '<sep>',
)


In [6]:
# compute 
# BLEU & ds-BLEU
compute_bleu_scores(ted_bleu, sys_fname_dict, model_list, to_realign = False)


Compute BLEU
Llama-3.1-8B-Instruct
EuroLLM-22B-Instruct
Qwen35-0.8B
Qwen35-9B
Compute ds-BLEU
15
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
15
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
15
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
15
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
15
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
15
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
15
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
15
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
15
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
12
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
12
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
12
signature = BLEU|nrefs:

In [6]:
show_bleu_scores(ted_bleu, sys_fname_dict, max_n_order)


BLEU
2014


,Llama,EuroLLM22,Qwen35-9B
sent,38.3 (1.0),45.5 (1.0),38.8 (1.0)
256,39.0 (1.0),46.2 (1.0),35.7 (0.99)
512,39.8 (1.0),46.0 (1.0),35.5 (0.98)
768,39.7 (1.0),46.0 (1.0),36.0 (0.98)
1024,37.0 (1.0),46.2 (1.0),35.8 (0.98)
1200,39.3 (0.99),46.2 (1.0),36.8 (0.97)
1600,39.2 (0.99),45.8 (0.99),37.5 (0.97)
2048,39.3 (0.99),46.0 (0.99),37.6 (0.97)
doc,39.0 (0.99),45.7 (1.0),39.6 (0.98)


\begin{tabular}{llll}
 & Llama & EuroLLM22 & Qwen35-9B \\
sent & 38.3 (1.0) & 45.5 (1.0) & 38.8 (1.0) \\
256 & 39.0 (1.0) & 46.2 (1.0) & 35.7 (0.99) \\
512 & 39.8 (1.0) & 46.0 (1.0) & 35.5 (0.98) \\
768 & 39.7 (1.0) & 46.0 (1.0) & 36.0 (0.98) \\
1024 & 37.0 (1.0) & 46.2 (1.0) & 35.8 (0.98) \\
1200 & 39.3 (0.99) & 46.2 (1.0) & 36.8 (0.97) \\
1600 & 39.2 (0.99) & 45.8 (0.99) & 37.5 (0.97) \\
2048 & 39.3 (0.99) & 46.0 (0.99) & 37.6 (0.97) \\
doc & 39.0 (0.99) & 45.7 (1.0) & 39.6 (0.98) \\
\end{tabular}

2015


,Llama,EuroLLM22,Qwen35-9B
sent,37.6 (1.0),42.5 (1.0),36.6 (1.0)
256,38.1 (1.0),43.2 (1.0),35.1 (1.0)
512,38.5 (1.0),43.7 (1.0),35.7 (1.0)
768,38.4 (1.0),43.6 (1.0),36.7 (1.0)
1024,38.6 (1.0),43.6 (1.0),36.7 (1.0)
1200,38.6 (1.0),43.4 (1.0),36.7 (1.0)
1600,38.5 (1.0),43.7 (1.0),37.2 (1.0)
2048,38.5 (1.0),43.8 (1.0),37.9 (1.0)
doc,38.7 (1.0),43.9 (1.0),38.1 (1.0)


\begin{tabular}{llll}
 & Llama & EuroLLM22 & Qwen35-9B \\
sent & 37.6 (1.0) & 42.5 (1.0) & 36.6 (1.0) \\
256 & 38.1 (1.0) & 43.2 (1.0) & 35.1 (1.0) \\
512 & 38.5 (1.0) & 43.7 (1.0) & 35.7 (1.0) \\
768 & 38.4 (1.0) & 43.6 (1.0) & 36.7 (1.0) \\
1024 & 38.6 (1.0) & 43.6 (1.0) & 36.7 (1.0) \\
1200 & 38.6 (1.0) & 43.4 (1.0) & 36.7 (1.0) \\
1600 & 38.5 (1.0) & 43.7 (1.0) & 37.2 (1.0) \\
2048 & 38.5 (1.0) & 43.8 (1.0) & 37.9 (1.0) \\
doc & 38.7 (1.0) & 43.9 (1.0) & 38.1 (1.0) \\
\end{tabular}

2016


...

\begin{tabular}{llll}
 & Llama & EuroLLM22 & Qwen35-9B \\
sent & 40.9 (1.0) & 45.9 (1.0) & 40.5 (1.0) \\
256 & 41.4 (1.0) & 46.6 (0.99) & 38.5 (0.99) \\
512 & 41.7 (0.99) & 47.1 (0.99) & 38.9 (0.99) \\
768 & 41.4 (0.99) & 47.0 (0.99) & 39.8 (0.99) \\
1024 & 41.7 (0.99) & 47.1 (0.99) & 39.5 (0.98) \\
1200 & 41.6 (0.99) & 46.9 (0.99) & 39.5 (0.98) \\
1600 & 41.6 (0.99) & 47.1 (0.99) & 40.2 (0.98) \\
2048 & 41.6 (0.99) & 47.3 (0.99) & 40.8 (0.98) \\
doc & 41.7 (0.99) & 47.4 (0.99) & 40.9 (0.98) \\
\end{tabular}

2016


,Llama,EuroLLM22,Qwen35-9B
sent,37.6 (1.0),43.1 (1.0),38.4 (1.0)
256,38.7 (1.0),43.8 (1.0),38.1 (1.0)
512,39.1 (1.0),43.7 (1.0),38.4 (1.0)
768,38.6 (1.0),43.4 (1.0),38.5 (1.0)
1024,39.1 (1.0),43.9 (1.0),38.2 (1.0)
1200,38.8 (1.0),43.5 (1.0),39.4 (1.0)
1600,39.0 (1.0),43.9 (1.0),39.4 (1.0)
2048,38.7 (1.0),43.5 (1.0),39.9 (1.0)
doc,38.8 (1.0),43.6 (1.0),40.5 (1.0)


\begin{tabular}{llll}
 & Llama & EuroLLM22 & Qwen35-9B \\
sent & 37.6 (1.0) & 43.1 (1.0) & 38.4 (1.0) \\
256 & 38.7 (1.0) & 43.8 (1.0) & 38.1 (1.0) \\
512 & 39.1 (1.0) & 43.7 (1.0) & 38.4 (1.0) \\
768 & 38.6 (1.0) & 43.4 (1.0) & 38.5 (1.0) \\
1024 & 39.1 (1.0) & 43.9 (1.0) & 38.2 (1.0) \\
1200 & 38.8 (1.0) & 43.5 (1.0) & 39.4 (1.0) \\
1600 & 39.0 (1.0) & 43.9 (1.0) & 39.4 (1.0) \\
2048 & 38.7 (1.0) & 43.5 (1.0) & 39.9 (1.0) \\
doc & 38.8 (1.0) & 43.6 (1.0) & 40.5 (1.0) \\
\end{tabular}

2017


,Llama,EuroLLM22,Qwen35-9B
sent,37.8 (1.0),43.5 (1.0),38.4 (1.0)
256,38.6 (1.0),44.0 (1.0),37.5 (1.0)
512,39.2 (1.0),43.8 (1.0),38.1 (1.0)
768,39.1 (1.0),44.0 (1.0),38.7 (1.0)
1024,38.9 (1.0),43.9 (1.0),38.4 (1.0)
1200,39.1 (1.0),44.0 (1.0),38.8 (1.0)
1600,39.0 (1.0),44.2 (1.0),39.1 (1.0)
2048,38.4 (1.0),44.1 (1.0),39.9 (1.0)
doc,38.7 (1.0),43.8 (1.0),41.1 (1.0)


\begin{tabular}{llll}
 & Llama & EuroLLM22 & Qwen35-9B \\
sent & 37.8 (1.0) & 43.5 (1.0) & 38.4 (1.0) \\
256 & 38.6 (1.0) & 44.0 (1.0) & 37.5 (1.0) \\
512 & 39.2 (1.0) & 43.8 (1.0) & 38.1 (1.0) \\
768 & 39.1 (1.0) & 44.0 (1.0) & 38.7 (1.0) \\
1024 & 38.9 (1.0) & 43.9 (1.0) & 38.4 (1.0) \\
1200 & 39.1 (1.0) & 44.0 (1.0) & 38.8 (1.0) \\
1600 & 39.0 (1.0) & 44.2 (1.0) & 39.1 (1.0) \\
2048 & 38.4 (1.0) & 44.1 (1.0) & 39.9 (1.0) \\
doc & 38.7 (1.0) & 43.8 (1.0) & 41.1 (1.0) \\
\end{tabular}



In [6]:
print('ds-BLEU')

table_ls = []
res_dict = ted_bleu.get_ds_bleu_corpus_dict(sys_fname_dict, show_bp = True, small_bp_size = False)
for subset in ted_bleu.get_sub_corpus_list():
    print(subset)
        
    ds_bleu_df = pd.DataFrame(res_dict[subset]).loc[max_n_order]
    ds_bleu_df.columns = pd.MultiIndex.from_product([[subset], ds_bleu_df.columns])
    table_ls.append(ds_bleu_df)

table_df = pd.concat(table_ls, axis = 1)
display(table_df)



table_ls = []
res_dict = ted_bleu.get_ds_bleu_corpus_dict(sys_fname_dict, show_bp = False)
for subset in ted_bleu.get_sub_corpus_list():
    print(subset)
    ds_bleu_df = pd.DataFrame(res_dict[subset]).loc[max_n_order]
    ds_bleu_df.columns = pd.MultiIndex.from_product([[subset], ds_bleu_df.columns])
    table_ls.append(ds_bleu_df)

table_df = pd.concat(table_ls, axis = 1)
display(table_df)
print(table_df.style.to_latex(column_format="l" + "r" * len(table_df.columns)))

ds-BLEU
2014
2015
2016
2017


2014                                   2015               \
            Llama    EuroLLM22    Qwen35-9B        Llama    EuroLLM22   
sent  41.1 (0.99)  47.9 (0.99)  41.8 (0.99)   40.9 (1.0)   45.9 (1.0)   
256   42.0 (0.99)  48.7 (0.99)  38.7 (0.98)   41.4 (1.0)  46.6 (0.99)   
512   42.7 (0.99)  48.3 (0.99)  38.4 (0.97)  41.7 (0.99)  47.1 (0.99)   
768   42.4 (0.99)  48.3 (0.99)  39.2 (0.97)  41.4 (0.99)  47.0 (0.99)   
1024  41.3 (0.99)  48.7 (0.99)  39.3 (0.97)  41.7 (0.99)  47.1 (0.99)   
1200  42.1 (0.98)  48.7 (0.99)  40.2 (0.97)  41.6 (0.99)  46.9 (0.99)   
1600  42.0 (0.98)  48.3 (0.98)  40.4 (0.97)  41.6 (0.99)  47.1 (0.99)   
2048  42.2 (0.98)  48.4 (0.98)  40.8 (0.97)  41.6 (0.99)  47.3 (0.99)   
doc   41.8 (0.98)  48.2 (0.99)  42.0 (0.97)  41.7 (0.99)  47.4 (0.99)   

                         2016                                2017              \
        Qwen35-9B       Llama   EuroLLM22   Qwen35-9B       Llama   EuroLLM22   
sent   40.5 (1.0)  37.6 (1.0)  43.1 (1.0)  38.4 (1.0)  37.8 (1.0)  43.5 (1.0)   
256   38.5 (0.99)  38.7 (1.0)  43.8 (1.0)  38.1 (1.0)  38.6 (1.0)  44.0 (1.0)   
512   38.9 (0.99)  39.1 (1.0)  43.7 (1.0)  38.4 (1.0)  39.2 (1.0)  43.8 (1.0)   
768   39.8 (0.99)  38.6 (1.0)  43.4 (1.0)  38.5 (1.0)  39.1 (1.0)  44.0 (1.0)   
1024  39.5 (0.98)  39.1 (1.0)  43.9 (1.0)  38.2 (1.0)  38.9 (1.0)  43.9 (1.0)   
1200  39.5 (0.98)  38.8 (1.0)  43.5 (1.0)  39.4 (1.0)  39.1 (1.0)  44.0 (1.0)   
1600  40.2 (0.98)  39.0 (1.0)  43.9 (1.0)  39.4 (1.0)  39.0 (1.0)  44.2 (1.0)   
2048  40.8 (0.98)  38.7 (1.0)  43.5 (1.0)  39.9 (1.0)  38.4 (1.0)  44.1 (1.0)   
doc   40.9 (0.98)  38.8 (1.0)  43.6 (1.0)  40.5 (1.0)  38.7 (1.0)  43.8 (1.0)   

                  
       Qwen35-9B  
sent  38.4 (1.0)  
256   37.5 (1.0)  
512   38.1 (1.0)  
768   38.7 (1.0)  
1024  38.4 (1.0)  
1200  38.8 (1.0)  
1600  39.1 (1.0)  
2048  39.9 (1.0)  
doc   41.1 (1.0)

2014
2015
2016
2017


2014                      2015                      2016            \
     Llama EuroLLM22 Qwen35-9B Llama EuroLLM22 Qwen35-9B Llama EuroLLM22   
sent  41.1      47.9      41.8  40.9      45.9      40.5  37.6      43.1   
256   42.0      48.7      38.7  41.4      46.6      38.5  38.7      43.8   
512   42.7      48.3      38.4  41.7      47.1      38.9  39.1      43.7   
768   42.4      48.3      39.2  41.4      47.0      39.8  38.6      43.4   
1024  41.3      48.7      39.3  41.7      47.1      39.5  39.1      43.9   
1200  42.1      48.7      40.2  41.6      46.9      39.5  38.8      43.5   
1600  42.0      48.3      40.4  41.6      47.1      40.2  39.0      43.9   
2048  42.2      48.4      40.8  41.6      47.3      40.8  38.7      43.5   
doc   41.8      48.2      42.0  41.7      47.4      40.9  38.8      43.6   

                2017                      
     Qwen35-9B Llama EuroLLM22 Qwen35-9B  
sent      38.4  37.8      43.5      38.4  
256       38.1  38.6      44.0      37.5  
512       38.4  39.2      43.8      38.1  
768       38.5  39.1      44.0      38.7  
1024      38.2  38.9      43.9      38.4  
1200      39.4  39.1      44.0      38.8  
1600      39.4  39.0      44.2      39.1  
2048      39.9  38.4      44.1      39.9  
doc       40.5  38.7      43.8      41.1

\begin{tabular}{lrrrrrrrrrrrr}
 & \multicolumn{3}{r}{2014} & \multicolumn{3}{r}{2015} & \multicolumn{3}{r}{2016} & \multicolumn{3}{r}{2017} \\
 & Llama & EuroLLM22 & Qwen35-9B & Llama & EuroLLM22 & Qwen35-9B & Llama & EuroLLM22 & Qwen35-9B & Llama & EuroLLM22 & Qwen35-9B \\
sent & 41.1 & 47.9 & 41.8 & 40.9 & 45.9 & 40.5 & 37.6 & 43.1 & 38.4 & 37.8 & 43.5 & 38.4 \\
256 & 42.0 & 48.7 & 38.7 & 41.4 & 46.6 & 38.5 & 38.7 & 43.8 & 38.1 & 38.6 & 44.0 & 37.5 \\
512 & 42.7 & 48.3 & 38.4 & 41.7 & 47.1 & 38.9 & 39.1 & 43.7 & 38.4 & 39.2 & 43.8 & 38.1 \\
768 & 42.4 & 48.3 & 39.2 & 41.4 & 47.0 & 39.8 & 38.6 & 43.4 & 38.5 & 39.1 & 44.0 & 38.7 \\
1024 & 41.3 & 48.7 & 39.3 & 41.7 & 47.1 & 39.5 & 39.1 & 43.9 & 38.2 & 38.9 & 43.9 & 38.4 \\
1200 & 42.1 & 48.7 & 40.2 & 41.6 & 46.9 & 39.5 & 38.8 & 43.5 & 39.4 & 39.1 & 44.0 & 38.8 \\
1600 & 42.0 & 48.3 & 40.4 & 41.6 & 47.1 & 40.2 & 39.0 & 43.9 & 39.4 & 39.0 & 44.2 & 39.1 \\
2048 & 42.2 & 48.4 & 40.8 & 41.6 & 47.3 & 40.8 & 38.7 & 43.5 & 39.9 & 38.4 & 44.1 & 

### 2. ds-BLEU with t-test

In [7]:
to_test_max_n = ['sent'] + sorted(list(valid_max_n)) + [ 'doc', 'sent', 'doc' ]

show_ttest_ds_bleu(ted_bleu, to_test_max_n, sys_fname_dict )


Llama
EuroLLM22
Qwen35-9B


,Llama,EuroLLM22,Qwen35-9B
sent-256,-0.8,-0.7,1.6
256-512,-0.5,-,-
512-768,-,-,-
768-1024,-,-,-
1024-1200,-,-,-
1200-1600,-,-,-
1600-2048,-,-,\mark{-0.6}
2048-doc,-,-,-0.8
doc-sent,0.9,0.6,1.3
sent-doc,-0.9,-0.6,-1.3


\begin{tabular}{llll}
 & Llama & EuroLLM22 & Qwen35-9B \\
sent-256 & -0.8 & -0.7 & 1.6 \\
256-512 & -0.5 & - & - \\
512-768 & - & - & - \\
768-1024 & - & - & - \\
1024-1200 & - & - & - \\
1200-1600 & - & - & - \\
1600-2048 & - & - & \mark{-0.6} \\
2048-doc & - & - & -0.8 \\
doc-sent & 0.9 & 0.6 & 1.3 \\
sent-doc & -0.9 & -0.6 & -1.3 \\
\end{tabular}

Llama
EuroLLM22
Qwen35-9B


,Llama,EuroLLM22,Qwen35-9B
sent-256,-0.8 \footnotesize{(0.0)},-0.7 \footnotesize{(0.0)},1.6 \footnotesize{(0.0)}
256-512,-0.5 \footnotesize{(0.0)},0.1 \footnotesize{(0.43)},-0.2 \footnotesize{(0.28)}
512-768,0.3 \footnotesize{(0.06)},0.1 \footnotesize{(0.69)},-0.6 \footnotesize{(0.06)}
768-1024,0.2 \footnotesize{(0.61)},-0.2 \footnotesize{(0.07)},0.2 \footnotesize{(0.62)}
1024-1200,-0.1 \footnotesize{(0.68)},0.1 \footnotesize{(0.42)},-0.6 \footnotesize{(0.06)}
1200-1600,-0.0 \footnotesize{(0.9)},-0.1 \footnotesize{(0.55)},-0.3 \footnotesize{(0.28)}
1600-2048,0.2 \footnotesize{(0.14)},0.1 \footnotesize{(0.55)},\mark{-0.6 \footnotesize{(0.05)}}
2048-doc,-0.0 \footnotesize{(0.88)},0.1 \footnotesize{(0.48)},-0.8 \footnotesize{(0.01)}
doc-sent,0.9 \footnotesize{(0.0)},0.6 \footnotesize{(0.0)},1.3 \footnotesize{(0.0)}
sent-doc,-0.9 \footnotesize{(0.0)},-0.6 \footnotesize{(0.0)},-1.3 \footnotesize{(0.0)}


### 3. COMET with t-test

In [5]:
tedComet = CometEvaluator(
    dataset_paths = ted_data_path,
    system_paths = ted_mt_path,
    valid_max_n = valid_max_n,
    corpus_name = corpus_name,
    sub_corpus_list = sub_corpus_list,
    sep_tag = '<sep>',
    )

order = ['sent'] + [str(v) for v in sorted(valid_max_n)] + ['doc']


In [9]:
# gather comet scores, compute ds-comet
for model_type in model_list:  
    for subset in tedComet.get_sub_corpus_list():
        tedComet.store_comet_score(model_type, subset=subset, is_json = False)
        tedComet.get_ds_comet_and_store(model_type, subset=subset)


In [6]:
# show COMET
comet_sent_dict = tedComet.get_comet_sent_dict(sys_fname_dict)

for k, v in comet_sent_dict.items():
    print(k)
    display(v)


Llama


,2014,2015,2016,2017
256,0.814158,0.824640,0.818751,0.814481
512,0.815587,0.824379,0.821461,0.815306
1600,0.810255,0.821520,0.821162,0.813395
768,0.813732,0.824135,0.819798,0.813963
1024,0.810598,0.823685,0.822579,0.813051
2048,0.811204,0.822907,0.821336,0.810006
doc,0.811552,0.821686,0.822841,0.811888
1200,0.810458,0.822184,0.818768,0.815442
sent,0.837932,0.844265,0.841706,0.841431


EuroLLM22


,2014,2015,2016,2017
256,0.848892,0.851330,0.848112,0.840420
512,0.847693,0.850583,0.845466,0.840452
1600,0.840667,0.847405,0.847501,0.839137
768,0.847185,0.849201,0.844081,0.840117
1024,0.846160,0.850408,0.845006,0.839045
2048,0.839431,0.849575,0.843278,0.837696
doc,0.841902,0.849048,0.844361,0.834644
1200,0.842709,0.849555,0.845151,0.838514
sent,0.863950,0.864060,0.866393,0.863141


Qwen35-9B


,2014,2015,2016,2017
256,0.827805,0.834561,0.834675,0.824482
512,0.824320,0.832452,0.835020,0.826213
1600,0.831132,0.836587,0.837760,0.825163
768,0.827167,0.835693,0.835564,0.829233
1024,0.825374,0.832905,0.835259,0.828280
2048,0.828264,0.839380,0.836883,0.830807
doc,0.834592,0.839637,0.843805,0.833354
1200,0.830889,0.838263,0.839512,0.825570
sent,0.852784,0.859171,0.859563,0.855867


In [23]:
res_dict = {}
for subset in tedComet.get_sub_corpus_list():
    for model_type in model_list:  
        tmp = rename_dict[model_type]
        tmp_dict = comet_sent_dict[tmp].to_dict()
    
        res_dict[(subset, tmp)] = tmp_dict[subset]

res_df = pd.DataFrame(res_dict).rename(columns = format_rename_dict, level = 1).loc[order]*100
display(res_df)
print(res_df.style.format(precision = 1).to_latex(column_format="l" + "r" * len(res_df.columns)))

2014                                              2015  \
          Llama EuroLLM$_{22}$ Qwen{3.5}$_{9\text{B}}$      Llama   
sent  83.793170      86.395028               85.278382  84.426516   
256   81.415785      84.889236               82.780526  82.464026   
512   81.558734      84.769259               82.432002  82.437880   
768   81.373240      84.718500               82.716734  82.413499   
1024  81.059820      84.616041               82.537395  82.368535   
1200  81.045794      84.270864               83.088928  82.218374   
1600  81.025518      84.066732               83.113195  82.152047   
2048  81.120419      83.943149               82.826423  82.290681   
doc   81.155187      84.190221               83.459210  82.168592   

                                                  2016                 \
     EuroLLM$_{22}$ Qwen{3.5}$_{9\text{B}}$      Llama EuroLLM$_{22}$   
sent      86.406044               85.917093  84.170559      86.639349   
256       85.133033               83.456137  81.875106      84.811172   
512       85.058319               83.245203  82.146064      84.546554   
768       84.920064               83.569335  81.979809      84.408140   
1024      85.040807               83.290514  82.257880      84.500636   
1200      84.955463               83.826326  81.876824      84.515094   
1600      84.740540               83.658699  82.116217      84.750096   
2048      84.957541               83.938047  82.133601      84.327777   
doc       84.904798               83.963676  82.284139      84.436117   

                                   2017                                         
     Qwen{3.5}$_{9\text{B}}$      Llama EuroLLM$_{22}$ Qwen{3.5}$_{9\text{B}}$  
sent               85.956259  84.143128      86.314065               85.586658  
256                83.467485  81.448057      84.042003               82.448193  
512                83.502006  81.530570      84.045204               82.621284  
768                83.556418  81.396334      84.011723               82.923306  
1024               83.525908  81.305119      83.904502               82.827999  
1200               83.951237  81.544167      83.851428               82.556966  
1600               83.775963  81.339472      83.913749               82.516318  
2048               83.688283  81.000620      83.769626               83.080659  
doc                84.380513  81.188813      83.464390               83.335446

\begin{tabular}{lrrrrrrrrrrrr}
 & \multicolumn{3}{r}{2014} & \multicolumn{3}{r}{2015} & \multicolumn{3}{r}{2016} & \multicolumn{3}{r}{2017} \\
 & Llama & EuroLLM$_{22}$ & Qwen{3.5}$_{9\text{B}}$ & Llama & EuroLLM$_{22}$ & Qwen{3.5}$_{9\text{B}}$ & Llama & EuroLLM$_{22}$ & Qwen{3.5}$_{9\text{B}}$ & Llama & EuroLLM$_{22}$ & Qwen{3.5}$_{9\text{B}}$ \\
sent & 83.8 & 86.4 & 85.3 & 84.4 & 86.4 & 85.9 & 84.2 & 86.6 & 86.0 & 84.1 & 86.3 & 85.6 \\
256 & 81.4 & 84.9 & 82.8 & 82.5 & 85.1 & 83.5 & 81.9 & 84.8 & 83.5 & 81.4 & 84.0 & 82.4 \\
512 & 81.6 & 84.8 & 82.4 & 82.4 & 85.1 & 83.2 & 82.1 & 84.5 & 83.5 & 81.5 & 84.0 & 82.6 \\
768 & 81.4 & 84.7 & 82.7 & 82.4 & 84.9 & 83.6 & 82.0 & 84.4 & 83.6 & 81.4 & 84.0 & 82.9 \\
1024 & 81.1 & 84.6 & 82.5 & 82.4 & 85.0 & 83.3 & 82.3 & 84.5 & 83.5 & 81.3 & 83.9 & 82.8 \\
1200 & 81.0 & 84.3 & 83.1 & 82.2 & 85.0 & 83.8 & 81.9 & 84.5 & 84.0 & 81.5 & 83.9 & 82.6 \\
1600 & 81.0 & 84.1 & 83.1 & 82.2 & 84.7 & 83.7 & 82.1 & 84.8 & 83.8 & 81.3 & 83.9 & 82.5 \\
2048 & 8

In [ ]:
# show COMET


table_ls = []
comet_sent_dict = tedComet.get_comet_sent_dict(sys_fname_dict)

for subset in ted_bleu.get_sub_corpus_list():
    print(subset)
    ds_bleu_df = pd.DataFrame(res_dict[subset]).loc[max_n_order]
    ds_bleu_df.columns = pd.MultiIndex.from_product([[subset], ds_bleu_df.columns])
    table_ls.append(ds_bleu_df)

table_df = pd.concat(table_ls, axis = 1)
display(table_df)
print(table_df.style.to_latex(column_format="l" + "r" * len(table_df.columns)))


In [10]:
# compute and report t-test with ds-comet
to_test_max_n = ['sent'] + sorted(list(valid_max_n)) + [ 'doc', 'sent', 'doc' ]
show_ttest_comet(tedComet, to_test_max_n, sys_fname_dict )


Llama
sent-256 2.3 TtestResult(statistic=np.float64(14.747980319057875), pvalue=np.float64(4.951151702597216e-20), df=np.int64(51))
256-512 -0.1 TtestResult(statistic=np.float64(-1.4978040735778828), pvalue=np.float64(0.1403484062172255), df=np.int64(51))
512-768 0.2 TtestResult(statistic=np.float64(1.3976686777925225), pvalue=np.float64(0.16826292444699234), df=np.int64(51))
768-1024 -0.0 TtestResult(statistic=np.float64(-0.006819774230584027), pvalue=np.float64(0.9945852567523614), df=np.int64(51))
1024-1200 0.1 TtestResult(statistic=np.float64(1.0428444856687593), pvalue=np.float64(0.3019385816141672), df=np.int64(51))
1200-1600 0.0 TtestResult(statistic=np.float64(0.19972110711697527), pvalue=np.float64(0.842492918238162), df=np.int64(51))
1600-2048 -0.0 TtestResult(statistic=np.float64(-0.24839929061592417), pvalue=np.float64(0.8048230077800671), df=np.int64(51))
2048-doc -0.1 TtestResult(statistic=np.float64(-0.826755690142409), pvalue=np.float64(0.41222700471701024), df=np.int64

,Llama,EuroLLM22,Qwen35-9B
sent-256,2.3,1.8,2.6
256-512,-,-,-
512-768,-,-,-
768-1024,-,-,-
1024-1200,-,-,-
1200-1600,-,-,-
1600-2048,-,-,-
2048-doc,-,-,\mark{-0.3}
doc-sent,-2.4,-2.2,-2.0
sent-doc,2.4,2.2,2.0


\begin{tabular}{llll}
 & Llama & EuroLLM22 & Qwen35-9B \\
sent-256 & 2.3 & 1.8 & 2.6 \\
256-512 & - & - & - \\
512-768 & - & - & - \\
768-1024 & - & - & - \\
1024-1200 & - & - & - \\
1200-1600 & - & - & - \\
1600-2048 & - & - & - \\
2048-doc & - & - & \mark{-0.3} \\
doc-sent & -2.4 & -2.2 & -2.0 \\
sent-doc & 2.4 & 2.2 & 2.0 \\
\end{tabular}

Llama
sent-256 2.3 TtestResult(statistic=np.float64(14.747980319057875), pvalue=np.float64(4.951151702597216e-20), df=np.int64(51))
256-512 -0.1 TtestResult(statistic=np.float64(-1.4978040735778828), pvalue=np.float64(0.1403484062172255), df=np.int64(51))
512-768 0.2 TtestResult(statistic=np.float64(1.3976686777925225), pvalue=np.float64(0.16826292444699234), df=np.int64(51))
768-1024 -0.0 TtestResult(statistic=np.float64(-0.006819774230584027), pvalue=np.float64(0.9945852567523614), df=np.int64(51))
1024-1200 0.1 TtestResult(statistic=np.float64(1.0428444856687593), pvalue=np.float64(0.3019385816141672), df=np.int64(51))
1200-1600 0.0 TtestResult

,Llama,EuroLLM22,Qwen35-9B
sent-256,2.3 \footnotesize{(0.0)},1.8 \footnotesize{(0.0)},2.6 \footnotesize{(0.0)}
256-512,-,-,-
512-768,-,-,-
768-1024,-,-,-
1024-1200,-,-,-
1200-1600,-,-,-
1600-2048,-,-,-
2048-doc,-,-,-0.3 \footnotesize{(0.04)}
doc-sent,-2.4 \footnotesize{(0.0)},-2.2 \footnotesize{(0.0)},-2.0 \footnotesize{(0.0)}
sent-doc,2.4 \footnotesize{(0.0)},2.2 \footnotesize{(0.0)},2.0 \footnotesize{(0.0)}


## 4. Position bias -- Corr(begin_id, comet) 

In [12]:
tedPos = PositionBiasAnalyzer(
    dataset_paths = ted_data_path,
    system_paths = ted_mt_path,
    valid_max_n = valid_max_n,
    corpus_name = corpus_name,
    sub_corpus_list = sub_corpus_list ,
    )

nb_positions = len(valid_max_n)
disable_full_doc = True

In [13]:
total = get_nb_sample_for_position_bias(tedPos, sys_fname_dict, nb_positions, disable_full_doc = disable_full_doc)
print(total)

set of different begin idx
2014, 190, maximum different positions = 7
2015, 138, maximum different positions = 7
2016, 158, maximum different positions = 7
2017, 308, maximum different positions = 7
2014, 190, maximum different positions = 7
2015, 138, maximum different positions = 7
2016, 158, maximum different positions = 7
2017, 308, maximum different positions = 7
2014, 190, maximum different positions = 7
2015, 138, maximum different positions = 7
2016, 158, maximum different positions = 7
2017, 308, maximum different positions = 7
794


In [14]:

compute_position_bias(tedPos, sys_fname_dict, nb_positions, disable_full_doc = disable_full_doc, show_p_val = False)

794
794
Comparing positon bias, make sure that models in sys_fname_dict share the same tokenizer
[60, 159, 241, 308, 539, 719, 1366]
Llama
794
794
Comparing positon bias, make sure that models in sys_fname_dict share the same tokenizer
[64, 169, 256, 328, 573, 765, 1452]
EuroLLM22
794
794
Comparing positon bias, make sure that models in sys_fname_dict share the same tokenizer
[61, 160, 242, 310, 542, 723, 1374]
Qwen35-9B


,index,Llama,index,EuroLLM22,index,Qwen35-9B
$p$0-$p$1,60-159,-,64-169,-,61-160,-
$p$1-$p$2,159-241,-,169-256,-,160-242,-
$p$2-$p$3,241-308,-,256-328,-,242-310,-
$p$3-$p$4,308-539,-,328-573,\mark{0.4},310-542,\mark{-0.6}
$p$4-$p$5,539-719,\mark{0.5},573-765,-,542-723,-
$p$5-$p$6,719-1366,-,765-1452,-,723-1374,-


\begin{tabular}{lllllll}
 & index & Llama & index & EuroLLM22 & index & Qwen35-9B \\
$p$0-$p$1 & 60-159 & - & 64-169 & - & 61-160 & - \\
$p$1-$p$2 & 159-241 & - & 169-256 & - & 160-242 & - \\
$p$2-$p$3 & 241-308 & - & 256-328 & - & 242-310 & - \\
$p$3-$p$4 & 308-539 & - & 328-573 & \mark{0.4} & 310-542 & \mark{-0.6} \\
$p$4-$p$5 & 539-719 & \mark{0.5} & 573-765 & - & 542-723 & - \\
$p$5-$p$6 & 719-1366 & - & 765-1452 & - & 723-1374 & - \\
\end{tabular}



,Llama,EuroLLM22,Qwen35-9B
$p$0-$p$1,-,-,-
$p$1-$p$2,-,-,-
$p$2-$p$3,-,-,-
$p$3-$p$4,-,\mark{0.4},\mark{-0.6}
$p$4-$p$5,\mark{0.5},-,-
$p$5-$p$6,-,-,-


\begin{tabular}{llll}
 & Llama & EuroLLM22 & Qwen35-9B \\
$p$0-$p$1 & - & - & - \\
$p$1-$p$2 & - & - & - \\
$p$2-$p$3 & - & - & - \\
$p$3-$p$4 & - & \mark{0.4} & \mark{-0.6} \\
$p$4-$p$5 & \mark{0.5} & - & - \\
$p$5-$p$6 & - & - & - \\
\end{tabular}



# StudentWindow

## 0. define paths

In [42]:
from data_paths import (
    StudentPaths,
    StudentSystemPaths,
)


valid_max_n =  {512, 1024, 1536, 2048, 3072, 4096}
max_n_order = ['sent'] + sorted(list(valid_max_n)) + ['doc']


corpus_name = 'STUDENT'

DATA_DIR = "/Users/ziqianpeng/MaTOS/data/STEP/corpora/txt/parallel_datasets/testsets"
student_data_path = StudentPaths(root = f'{DATA_DIR}/STUDENT')

SYS_DIR = "/Users/ziqianpeng/MaTOS/analyse_result/data_res/TED/eval_llm_output-window_update"
student_mt_path = StudentSystemPaths(root = f'{SYS_DIR}/STUDENT')

Supported max_n values: {512, 1024, 1536, 3072, 2048, 4096, 'doc', 'sent'}
Supported max_n values: {512, 1024, 1536, 3072, 2048, 4096, 'doc', 'sent'}


In [29]:
ngram_df = get_repeat_stat(
    sys_fname_dict, 
    student_data_path,
    student_mt_path, 
    sub_corpus_list,
    n = 10,
    verbose = False,
    min_repeat = 1, 
)

display(ngram_df.round(2))
print(ngram_df.style.format(precision = 2).to_latex(column_format="l" + "r" * len(ngram_df.columns)))

ratio of document containing 10gram


,512,1024,1536,3072,2048,4096,doc
ref,0.05,0.15,0.24,0.32,0.26,0.47,0.6
Llama,0.06,0.21,0.22,0.40,0.31,0.63,0.7
EuroLLM22,0.05,0.16,0.30,0.40,0.29,0.47,0.5
Qwen35-9B,0.07,0.15,0.20,0.40,0.26,0.42,0.7


\begin{tabular}{lrrrrrrr}
 & 512 & 1024 & 1536 & 3072 & 2048 & 4096 & doc \\
ref & 0.05 & 0.15 & 0.24 & 0.32 & 0.26 & 0.47 & 0.60 \\
Llama & 0.06 & 0.21 & 0.22 & 0.40 & 0.31 & 0.63 & 0.70 \\
EuroLLM22 & 0.05 & 0.16 & 0.30 & 0.40 & 0.29 & 0.47 & 0.50 \\
Qwen35-9B & 0.07 & 0.15 & 0.20 & 0.40 & 0.26 & 0.42 & 0.70 \\
\end{tabular}



In [30]:
ngram_df = get_repeat_stat(
    sys_fname_dict, 
    student_data_path,
    student_mt_path, 
    sub_corpus_list,
    n = 20,
    verbose = False,
    min_repeat = 1, 
)

display(ngram_df.round(2))
print(ngram_df.style.format(precision = 2).to_latex(column_format="l" + "r" * len(ngram_df.columns)))

ratio of document containing 20gram


,512,1024,1536,3072,2048,4096,doc
ref,0.01,0.03,0.07,0.12,0.06,0.11,0.4
Llama,0.01,0.04,0.07,0.08,0.06,0.11,0.5
EuroLLM22,0.01,0.03,0.07,0.16,0.09,0.16,0.5
Qwen35-9B,0.01,0.03,0.04,0.16,0.06,0.11,0.2


\begin{tabular}{lrrrrrrr}
 & 512 & 1024 & 1536 & 3072 & 2048 & 4096 & doc \\
ref & 0.01 & 0.03 & 0.07 & 0.12 & 0.06 & 0.11 & 0.40 \\
Llama & 0.01 & 0.04 & 0.07 & 0.08 & 0.06 & 0.11 & 0.50 \\
EuroLLM22 & 0.01 & 0.03 & 0.07 & 0.16 & 0.09 & 0.16 & 0.50 \\
Qwen35-9B & 0.01 & 0.03 & 0.04 & 0.16 & 0.06 & 0.11 & 0.20 \\
\end{tabular}



## 1. BLEU

In [16]:

student_bleu = BleuEvaluator(
    dataset_paths = student_data_path,
    system_paths = student_mt_path,
    valid_max_n = valid_max_n,
    corpus_name = corpus_name,
    sub_corpus_list = None,
    sep_tag = '<sep>',
)

In [6]:

# compute 
# BLEU & ds-BLEU
compute_bleu_scores(student_bleu, sys_fname_dict, model_list, to_realign = True)



Compute BLEU
Llama-3.1-8B-Instruct
EuroLLM-22B-Instruct
Qwen35-0.8B
Qwen35-9B
Compute ds-BLEU
10
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
10
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
10
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
10
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
10
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
10
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
10
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
10
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
10
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
10
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
10
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
10
signature = BLEU|nrefs:

In [17]:
# show BLEU & ds-BLEU scores
show_bleu_scores(student_bleu, sys_fname_dict, max_n_order)

BLEU
STUDENT


,Llama,EuroLLM22,Qwen35-9B
sent,51.9 (0.99),59.4 (0.98),50.8 (0.99)
512,52.3 (0.98),58.6 (0.98),48.3 (0.96)
1024,50.9 (0.97),57.6 (0.96),47.7 (0.94)
1536,49.9 (0.95),58.0 (0.97),47.2 (0.91)
2048,47.9 (0.92),52.7 (1.0),49.1 (0.92)
3072,46.1 (0.93),56.1 (0.95),48.2 (0.93)
4096,45.9 (0.88),52.8 (0.92),48.9 (0.91)
doc,31.7 (1.0),43.9 (0.9),49.0 (0.9)


\begin{tabular}{llll}
 & Llama & EuroLLM22 & Qwen35-9B \\
sent & 51.9 (0.99) & 59.4 (0.98) & 50.8 (0.99) \\
512 & 52.3 (0.98) & 58.6 (0.98) & 48.3 (0.96) \\
1024 & 50.9 (0.97) & 57.6 (0.96) & 47.7 (0.94) \\
1536 & 49.9 (0.95) & 58.0 (0.97) & 47.2 (0.91) \\
2048 & 47.9 (0.92) & 52.7 (1.0) & 49.1 (0.92) \\
3072 & 46.1 (0.93) & 56.1 (0.95) & 48.2 (0.93) \\
4096 & 45.9 (0.88) & 52.8 (0.92) & 48.9 (0.91) \\
doc & 31.7 (1.0) & 43.9 (0.9) & 49.0 (0.9) \\
\end{tabular}

ds-BLEU
STUDENT


,Llama,EuroLLM22,Qwen35-9B
sent,54.8 (0.98),62.3 (0.98),54.8 (0.98)
512,55.6 (0.97),61.4 (0.97),51.9 (0.95)
1024,54.4 (0.96),61.0 (0.96),51.6 (0.93)
1536,53.7 (0.95),60.9 (0.97),51.5 (0.91)
2048,51.2 (0.91),54.6 (0.94),52.7 (0.92)
3072,50.6 (0.92),59.4 (0.95),52.4 (0.92)
4096,50.5 (0.89),55.8 (0.91),52.7 (0.91)
doc,36.8 (0.81),46.9 (0.82),52.4 (0.9)


\begin{tabular}{llll}
 & Llama & EuroLLM22 & Qwen35-9B \\
sent & 54.8 (0.98) & 62.3 (0.98) & 54.8 (0.98) \\
512 & 55.6 (0.97) & 61.4 (0.97) & 51.9 (0.95) \\
1024 & 54.4 (0.96) & 61.0 (0.96) & 51.6 (0.93) \\
1536 & 53.7 (0.95) & 60.9 (0.97) & 51.5 (0.91) \\
2048 & 51.2 (0.91) & 54.6 (0.94) & 52.7 (0.92) \\
3072 & 50.6 (0.92) & 59.4 (0.95) & 52.4 (0.92) \\
4096 & 50.5 (0.89) & 55.8 (0.91) & 52.7 (0.91) \\
doc & 36.8 (0.81) & 46.9 (0.82) & 52.4 (0.9) \\
\end{tabular}



In [17]:

table_ls = []
res_dict = student_bleu.get_ds_bleu_corpus_dict(sys_fname_dict, show_bp = True, small_bp_size = True)
for subset in student_bleu.get_sub_corpus_list():
    print(subset)
    ds_bleu_df = pd.DataFrame(res_dict[subset]).loc[max_n_order]
    ds_bleu_df.columns = pd.MultiIndex.from_product([[subset], ds_bleu_df.columns])
    table_ls.append(ds_bleu_df)

student_table_df = pd.concat(table_ls, axis = 1)
display(student_table_df)
print(student_table_df.style.to_latex(column_format="l" + "r" * len(student_table_df.columns)))

STUDENT


STUDENT                              \
                           Llama                   EuroLLM22   
sent  54.8 \footnotesize{(0.98)}  62.3 \footnotesize{(0.98)}   
512   55.6 \footnotesize{(0.97)}  61.4 \footnotesize{(0.97)}   
1024  54.4 \footnotesize{(0.96)}  61.0 \footnotesize{(0.96)}   
1536  53.7 \footnotesize{(0.95)}  60.9 \footnotesize{(0.97)}   
2048  51.2 \footnotesize{(0.91)}  54.6 \footnotesize{(0.94)}   
3072  50.6 \footnotesize{(0.92)}  59.4 \footnotesize{(0.95)}   
4096  50.5 \footnotesize{(0.89)}  55.8 \footnotesize{(0.91)}   
doc   36.8 \footnotesize{(0.81)}  46.9 \footnotesize{(0.82)}   

                                  
                       Qwen35-9B  
sent  54.8 \footnotesize{(0.98)}  
512   51.9 \footnotesize{(0.95)}  
1024  51.6 \footnotesize{(0.93)}  
1536  51.5 \footnotesize{(0.91)}  
2048  52.7 \footnotesize{(0.92)}  
3072  52.4 \footnotesize{(0.92)}  
4096  52.7 \footnotesize{(0.91)}  
doc    52.4 \footnotesize{(0.9)}

\begin{tabular}{lrrr}
 & \multicolumn{3}{r}{STUDENT} \\
 & Llama & EuroLLM22 & Qwen35-9B \\
sent & 54.8 \footnotesize{(0.98)} & 62.3 \footnotesize{(0.98)} & 54.8 \footnotesize{(0.98)} \\
512 & 55.6 \footnotesize{(0.97)} & 61.4 \footnotesize{(0.97)} & 51.9 \footnotesize{(0.95)} \\
1024 & 54.4 \footnotesize{(0.96)} & 61.0 \footnotesize{(0.96)} & 51.6 \footnotesize{(0.93)} \\
1536 & 53.7 \footnotesize{(0.95)} & 60.9 \footnotesize{(0.97)} & 51.5 \footnotesize{(0.91)} \\
2048 & 51.2 \footnotesize{(0.91)} & 54.6 \footnotesize{(0.94)} & 52.7 \footnotesize{(0.92)} \\
3072 & 50.6 \footnotesize{(0.92)} & 59.4 \footnotesize{(0.95)} & 52.4 \footnotesize{(0.92)} \\
4096 & 50.5 \footnotesize{(0.89)} & 55.8 \footnotesize{(0.91)} & 52.7 \footnotesize{(0.91)} \\
doc & 36.8 \footnotesize{(0.81)} & 46.9 \footnotesize{(0.82)} & 52.4 \footnotesize{(0.9)} \\
\end{tabular}



## 2. ds-BLEU with t-test

In [18]:
to_test_max_n = ['sent'] + sorted(list(valid_max_n)) + [ 'doc', 'sent', 'doc' ]


show_ttest_ds_bleu(student_bleu, to_test_max_n, sys_fname_dict )

Llama
EuroLLM22
Qwen35-9B


,Llama,EuroLLM22,Qwen35-9B
sent-512,-,0.9,\mark{2.9}
512-1024,\mark{1.2},-,-
1024-1536,-,-,-
1536-2048,\mark{2.4},\mark{6.4},-
2048-3072,-,-,-
3072-4096,-,-,-
4096-doc,\mark{13.7},-,-
doc-sent,\mark{-18.0},\mark{-15.4},-
sent-doc,\mark{18.0},\mark{15.4},-


\begin{tabular}{llll}
 & Llama & EuroLLM22 & Qwen35-9B \\
sent-512 & - & 0.9 & \mark{2.9} \\
512-1024 & \mark{1.2} & - & - \\
1024-1536 & - & - & - \\
1536-2048 & \mark{2.4} & \mark{6.4} & - \\
2048-3072 & - & - & - \\
3072-4096 & - & - & - \\
4096-doc & \mark{13.7} & - & - \\
doc-sent & \mark{-18.0} & \mark{-15.4} & - \\
sent-doc & \mark{18.0} & \mark{15.4} & - \\
\end{tabular}

Llama
EuroLLM22
Qwen35-9B


,Llama,EuroLLM22,Qwen35-9B
sent-512,-0.7 \footnotesize{(0.08)},0.9 \footnotesize{(0.01)},\mark{2.9 \footnotesize{(0.01)}}
512-1024,\mark{1.2 \footnotesize{(0.04)}},0.4 \footnotesize{(0.37)},0.3 \footnotesize{(0.7)}
1024-1536,0.7 \footnotesize{(0.23)},0.1 \footnotesize{(0.86)},0.2 \footnotesize{(0.87)}
1536-2048,\mark{2.4 \footnotesize{(0.05)}},\mark{6.4 \footnotesize{(0.04)}},-1.2 \footnotesize{(0.09)}
2048-3072,0.7 \footnotesize{(0.75)},-4.8 \footnotesize{(0.06)},0.3 \footnotesize{(0.73)}
3072-4096,0.0 \footnotesize{(0.99)},3.6 \footnotesize{(0.07)},-0.3 \footnotesize{(0.76)}
4096-doc,\mark{13.7 \footnotesize{(0.04)}},8.9 \footnotesize{(0.16)},0.3 \footnotesize{(0.53)}
doc-sent,\mark{-18.0 \footnotesize{(0.02)}},\mark{-15.4 \footnotesize{(0.04)}},-2.5 \footnotesize{(0.08)}
sent-doc,\mark{18.0 \footnotesize{(0.02)}},\mark{15.4 \footnotesize{(0.04)}},2.5 \footnotesize{(0.08)}


## 3. COMET with t-test

In [35]:
studentComet = CometEvaluator(
    dataset_paths = student_data_path,
    system_paths = student_mt_path,
    valid_max_n = valid_max_n,
    corpus_name = corpus_name,
    sub_corpus_list = None,
    sep_tag = '<sep>',
    )


assert len(studentComet.get_sub_corpus_list()) == 1
subset = corpus_name


In [15]:
# gather comet scores, compute ds-comet
for model_type in model_list:  
    for subset in studentComet.get_sub_corpus_list():
        studentComet.store_comet_score(model_type, subset=subset, is_json = False)
        studentComet.get_ds_comet_and_store(model_type, subset=subset)


In [36]:
# show COMET
comet_sent_dict = studentComet.get_comet_sent_dict(sys_fname_dict)


order = ['sent'] + [str(v) for v in sorted(valid_max_n)] + ['doc']
comet_df = pd.concat([ v.rename(columns = {subset: k}) for k, v in comet_sent_dict.items() ], axis = 1).loc[order]


# display(comet_df.round(2))
# print(comet_df.style.format(precision=2).to_latex())

student_comet_df = comet_df*100
student_comet_df.columns = pd.MultiIndex.from_product([[subset], student_comet_df.columns])
student_comet_df
# ds_bleu_df.columns = pd.MultiIndex.from_product([[subset], ds_bleu_df.columns])

STUDENT                      
          Llama  EuroLLM22  Qwen35-9B
sent  83.948756  85.965305  85.495267
512   83.151425  85.067541  83.862023
1024  82.600567  84.555943  83.118286
1536  81.346566  84.648394  82.018297
2048  80.327004  81.855257  82.518110
3072  78.500209  83.856370  82.669234
4096  77.895796  81.682358  82.445143
doc   66.368625  75.970782  81.995448

In [21]:
to_test_max_n = ['sent'] + sorted(list(valid_max_n)) + [ 'doc', 'sent', 'doc' ]


show_ttest_comet(studentComet, to_test_max_n, sys_fname_dict )

Llama
sent-512 0.5 TtestResult(statistic=np.float64(1.8317079384420993), pvalue=np.float64(0.1002234059736006), df=np.int64(9))
512-1024 0.8 TtestResult(statistic=np.float64(2.2631479820969544), pvalue=np.float64(0.049919091083539036), df=np.int64(9))
1024-1536 1.1 TtestResult(statistic=np.float64(1.2773293385543214), pvalue=np.float64(0.23345147855728043), df=np.int64(9))
1536-2048 1.9 TtestResult(statistic=np.float64(1.4205032507300226), pvalue=np.float64(0.1891646454397208), df=np.int64(9))
2048-3072 0.5 TtestResult(statistic=np.float64(0.29656622090455487), pvalue=np.float64(0.7735276511476712), df=np.int64(9))
3072-4096 0.7 TtestResult(statistic=np.float64(0.4891543763903196), pvalue=np.float64(0.6364320847046996), df=np.int64(9))
4096-doc 8.8 TtestResult(statistic=np.float64(2.00072180657575), pvalue=np.float64(0.07646378558770978), df=np.int64(9))
doc-sent -14.4 TtestResult(statistic=np.float64(-2.7845257510521884), pvalue=np.float64(0.021243043789267), df=np.int64(9))
sent-doc 

,Llama,EuroLLM22,Qwen35-9B
sent-512,-,-,-
512-1024,\mark{0.8},-,-
1024-1536,-,-,-
1536-2048,-,\mark{3.0},-
2048-3072,-,-,-
3072-4096,-,-,-
4096-doc,-,-,-
doc-sent,\mark{-14.4},\mark{-9.7},\mark{-3.8}
sent-doc,\mark{14.4},\mark{9.7},\mark{3.8}


\begin{tabular}{llll}
 & Llama & EuroLLM22 & Qwen35-9B \\
sent-512 & - & - & - \\
512-1024 & \mark{0.8} & - & - \\
1024-1536 & - & - & - \\
1536-2048 & - & \mark{3.0} & - \\
2048-3072 & - & - & - \\
3072-4096 & - & - & - \\
4096-doc & - & - & - \\
doc-sent & \mark{-14.4} & \mark{-9.7} & \mark{-3.8} \\
sent-doc & \mark{14.4} & \mark{9.7} & \mark{3.8} \\
\end{tabular}

Llama
sent-512 0.5 TtestResult(statistic=np.float64(1.8317079384420993), pvalue=np.float64(0.1002234059736006), df=np.int64(9))
512-1024 0.8 TtestResult(statistic=np.float64(2.2631479820969544), pvalue=np.float64(0.049919091083539036), df=np.int64(9))
1024-1536 1.1 TtestResult(statistic=np.float64(1.2773293385543214), pvalue=np.float64(0.23345147855728043), df=np.int64(9))
1536-2048 1.9 TtestResult(statistic=np.float64(1.4205032507300226), pvalue=np.float64(0.1891646454397208), df=np.int64(9))
2048-3072 0.5 TtestResult(statistic=np.float64(0.29656622090455487), pvalue=np.float64(0.7735276511476712), df=np.int64(9))
3072-40

,Llama,EuroLLM22,Qwen35-9B
sent-512,-,-,-
512-1024,0.8 \footnotesize{(0.05)},-,-
1024-1536,-,-,-
1536-2048,-,3.0 \footnotesize{(0.03)},-
2048-3072,-,-,-
3072-4096,-,-,-
4096-doc,-,-,-
doc-sent,-14.4 \footnotesize{(0.02)},-9.7 \footnotesize{(0.04)},-3.8 \footnotesize{(0.02)}
sent-doc,14.4 \footnotesize{(0.02)},9.7 \footnotesize{(0.04)},3.8 \footnotesize{(0.02)}


## 4. Position bias -- Corr(begin_id, comet) 

In [22]:
studentPos = PositionBiasAnalyzer(
    dataset_paths = student_data_path,
    system_paths = student_mt_path,
    valid_max_n = valid_max_n,
    corpus_name = corpus_name,
    sub_corpus_list = None,
    )

assert len(studentPos.get_sub_corpus_list()) == 1
subset = corpus_name


In [23]:
disable_full_doc = False
nb_positions = len(valid_max_n)+1
total = get_nb_sample_for_position_bias(studentPos, sys_fname_dict, nb_positions, disable_full_doc = disable_full_doc)
print(total)
compute_position_bias(studentPos, sys_fname_dict, nb_positions, disable_full_doc = disable_full_doc, show_p_val = False)

set of different begin idx
STUDENT, 244, maximum different positions = 7
STUDENT, 244, maximum different positions = 7
STUDENT, 244, maximum different positions = 7
244
244
244
Comparing positon bias, make sure that models in sys_fname_dict share the same tokenizer
[162, 441, 847, 1165, 1340, 2538, 6268]
Llama
244
244
Comparing positon bias, make sure that models in sys_fname_dict share the same tokenizer
[180, 486, 926, 1278, 1469, 2782, 6893]
EuroLLM22
244
244
Comparing positon bias, make sure that models in sys_fname_dict share the same tokenizer
[171, 462, 881, 1216, 1397, 2650, 6550]
Qwen35-9B


,index,Llama,index,EuroLLM22,index,Qwen35-9B
$p$0-$p$1,162-441,-,180-486,\mark{1.3},171-462,-
$p$1-$p$2,441-847,\mark{1.3},486-926,-,462-881,-
$p$2-$p$3,847-1165,-,926-1278,\mark{-1.3},881-1216,-
$p$3-$p$4,1165-1340,2.5,1278-1469,2.1,1216-1397,-
$p$4-$p$5,1340-2538,\mark{1.5},1469-2782,\mark{1.4},1397-2650,\mark{1.2}
$p$5-$p$6,2538-6268,19.9,2782-6893,14.1,2650-6550,-


\begin{tabular}{lllllll}
 & index & Llama & index & EuroLLM22 & index & Qwen35-9B \\
$p$0-$p$1 & 162-441 & - & 180-486 & \mark{1.3} & 171-462 & - \\
$p$1-$p$2 & 441-847 & \mark{1.3} & 486-926 & - & 462-881 & - \\
$p$2-$p$3 & 847-1165 & - & 926-1278 & \mark{-1.3} & 881-1216 & - \\
$p$3-$p$4 & 1165-1340 & 2.5 & 1278-1469 & 2.1 & 1216-1397 & - \\
$p$4-$p$5 & 1340-2538 & \mark{1.5} & 1469-2782 & \mark{1.4} & 1397-2650 & \mark{1.2} \\
$p$5-$p$6 & 2538-6268 & 19.9 & 2782-6893 & 14.1 & 2650-6550 & - \\
\end{tabular}



,Llama,EuroLLM22,Qwen35-9B
$p$0-$p$1,-,\mark{1.3},-
$p$1-$p$2,\mark{1.3},-,-
$p$2-$p$3,-,\mark{-1.3},-
$p$3-$p$4,2.5,2.1,-
$p$4-$p$5,\mark{1.5},\mark{1.4},\mark{1.2}
$p$5-$p$6,19.9,14.1,-


\begin{tabular}{llll}
 & Llama & EuroLLM22 & Qwen35-9B \\
$p$0-$p$1 & - & \mark{1.3} & - \\
$p$1-$p$2 & \mark{1.3} & - & - \\
$p$2-$p$3 & - & \mark{-1.3} & - \\
$p$3-$p$4 & 2.5 & 2.1 & - \\
$p$4-$p$5 & \mark{1.5} & \mark{1.4} & \mark{1.2} \\
$p$5-$p$6 & 19.9 & 14.1 & - \\
\end{tabular}



In [24]:
disable_full_doc = True
nb_positions = len(valid_max_n)
total = get_nb_sample_for_position_bias(studentPos, sys_fname_dict, nb_positions, disable_full_doc = disable_full_doc)
print(total)
compute_position_bias(studentPos, sys_fname_dict, nb_positions, disable_full_doc = disable_full_doc, show_p_val = False)

set of different begin idx
STUDENT, 265, maximum different positions = 6
STUDENT, 265, maximum different positions = 6
STUDENT, 265, maximum different positions = 6
265
265
265
Comparing positon bias, make sure that models in sys_fname_dict share the same tokenizer
[164, 448, 827, 1132, 1360, 2632]
Llama
265
265
Comparing positon bias, make sure that models in sys_fname_dict share the same tokenizer
[182, 495, 906, 1242, 1493, 2885]
EuroLLM22
265
265
Comparing positon bias, make sure that models in sys_fname_dict share the same tokenizer
[173, 469, 861, 1181, 1418, 2744]
Qwen35-9B


,index,Llama,index,EuroLLM22,index,Qwen35-9B
$p$0-$p$1,164-448,-,182-495,\mark{1.2},173-469,-
$p$1-$p$2,448-827,-,495-906,-,469-861,-
$p$2-$p$3,827-1132,-,906-1242,\mark{-1.2},861-1181,-
$p$3-$p$4,1132-1360,\mark{1.9},1242-1493,1.7,1181-1418,-
$p$4-$p$5,1360-2632,3.7,1493-2885,3.7,1418-2744,\mark{1.1}


\begin{tabular}{lllllll}
 & index & Llama & index & EuroLLM22 & index & Qwen35-9B \\
$p$0-$p$1 & 164-448 & - & 182-495 & \mark{1.2} & 173-469 & - \\
$p$1-$p$2 & 448-827 & - & 495-906 & - & 469-861 & - \\
$p$2-$p$3 & 827-1132 & - & 906-1242 & \mark{-1.2} & 861-1181 & - \\
$p$3-$p$4 & 1132-1360 & \mark{1.9} & 1242-1493 & 1.7 & 1181-1418 & - \\
$p$4-$p$5 & 1360-2632 & 3.7 & 1493-2885 & 3.7 & 1418-2744 & \mark{1.1} \\
\end{tabular}



,Llama,EuroLLM22,Qwen35-9B
$p$0-$p$1,-,\mark{1.2},-
$p$1-$p$2,-,-,-
$p$2-$p$3,-,\mark{-1.2},-
$p$3-$p$4,\mark{1.9},1.7,-
$p$4-$p$5,3.7,3.7,\mark{1.1}


\begin{tabular}{llll}
 & Llama & EuroLLM22 & Qwen35-9B \\
$p$0-$p$1 & - & \mark{1.2} & - \\
$p$1-$p$2 & - & - & - \\
$p$2-$p$3 & - & \mark{-1.2} & - \\
$p$3-$p$4 & \mark{1.9} & 1.7 & - \\
$p$4-$p$5 & 3.7 & 3.7 & \mark{1.1} \\
\end{tabular}



# MersenneWindow


In [44]:
from data_paths import (
    MersennePaths,
    MersenneSystemPaths,
)

######################################
valid_max_n =  {512, 1024, 1536, 2048, 3072,  4096}
max_n_order = ['sent'] + sorted(list(valid_max_n)) + ['doc']

corpus_name = 'MERSENNE-GEOS'

DATA_DIR = "/Users/ziqianpeng/MaTOS/data/STEP/corpora/txt/parallel_datasets/testsets"
mersenne_data_path = MersennePaths(root = f'{DATA_DIR}/MERSENNE-GEOS')

SYS_DIR = "/Users/ziqianpeng/MaTOS/analyse_result/data_res/TED/eval_llm_output-window_update"
mersenne_mt_path = MersenneSystemPaths(root = f'{SYS_DIR}/MERSENNE-GEOS')




Supported max_n values: {512, 1024, 1536, 3072, 2048, 4096, 'doc', 'sent'}
Supported max_n values: {512, 1024, 1536, 3072, 2048, 4096, 'doc', 'sent'}


In [45]:
ngram_df = get_repeat_stat(
    sys_fname_dict, 
    mersenne_data_path,
    mersenne_mt_path, 
    sub_corpus_list,
    n = 10,
    verbose = False,
    min_repeat = 1, 
)

display(ngram_df.round(4))
print(ngram_df.style.format(precision = 2).to_latex(column_format="l" + "r" * len(ngram_df.columns)))

ngram_df = get_repeat_stat(
    sys_fname_dict, 
    mersenne_data_path,
    mersenne_mt_path, 
    sub_corpus_list,
    n = 10,
    verbose = False,
    min_repeat = 2, 
)

display(ngram_df.round(4))
print(ngram_df.style.format(precision = 2).to_latex(column_format="l" + "r" * len(ngram_df.columns)))

ratio of document containing 10gram


,512,1024,1536,3072,2048,4096,doc
ref,0.0150,0.0746,0.0870,0.2449,0.1507,0.2927,0.5263
Llama,0.0188,0.0746,0.0978,0.3265,0.2192,0.3415,0.6316
EuroLLM22,0.0188,0.0970,0.0978,0.2653,0.1918,0.4634,0.6842
Qwen35-9B,0.0113,0.0597,0.0761,0.2653,0.1370,0.3171,0.5789


\begin{tabular}{lrrrrrrr}
 & 512 & 1024 & 1536 & 3072 & 2048 & 4096 & doc \\
ref & 0.02 & 0.07 & 0.09 & 0.24 & 0.15 & 0.29 & 0.53 \\
Llama & 0.02 & 0.07 & 0.10 & 0.33 & 0.22 & 0.34 & 0.63 \\
EuroLLM22 & 0.02 & 0.10 & 0.10 & 0.27 & 0.19 & 0.46 & 0.68 \\
Qwen35-9B & 0.01 & 0.06 & 0.08 & 0.27 & 0.14 & 0.32 & 0.58 \\
\end{tabular}

ratio of document containing 10gram


,512,1024,1536,3072,2048,4096,doc
ref,0.0000,0.0,0.0,0.0000,0.0000,0.0000,0.0526
Llama,0.0000,0.0,0.0,0.0000,0.0000,0.0000,0.1053
EuroLLM22,0.0000,0.0,0.0,0.0204,0.0137,0.0488,0.2105
Qwen35-9B,0.0038,0.0,0.0,0.0000,0.0000,0.0244,0.1579


\begin{tabular}{lrrrrrrr}
 & 512 & 1024 & 1536 & 3072 & 2048 & 4096 & doc \\
ref & 0.00 & 0.00 & 0.00 & 0.00 & 0.00 & 0.00 & 0.05 \\
Llama & 0.00 & 0.00 & 0.00 & 0.00 & 0.00 & 0.00 & 0.11 \\
EuroLLM22 & 0.00 & 0.00 & 0.00 & 0.02 & 0.01 & 0.05 & 0.21 \\
Qwen35-9B & 0.00 & 0.00 & 0.00 & 0.00 & 0.00 & 0.02 & 0.16 \\
\end{tabular}



In [47]:

# ngram_df = get_repeat_stat(
#     sys_fname_dict, 
#     mersenne_data_path,
#     mersenne_mt_path, 
#     sub_corpus_list,
#     n = 10,
#     verbose = True,
#     min_repeat = 2, 
# )

### 1.BLEU

In [19]:

######################################
mersenne_bleu = BleuEvaluator(
    dataset_paths = mersenne_data_path,
    system_paths  = mersenne_mt_path,
    valid_max_n = valid_max_n,
    corpus_name = corpus_name,
    sub_corpus_list = None,
    sep_tag = '<sep>',
)


In [30]:
# compute 
# BLEU & ds-BLEU
compute_bleu_scores(mersenne_bleu, sys_fname_dict, model_list, to_realign = True)


Compute BLEU
Llama-3.1-8B-Instruct
EuroLLM-22B-Instruct
Qwen35-0.8B
Qwen35-9B
Compute ds-BLEU
19
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
19
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
19
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
19
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
19
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
19
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
19
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
19
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
19
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
19
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
19
signature = BLEU|nrefs:1|case:mixed|eff:yes|tok:13a|smooth:exp|version:2.4.0
19
signature = BLEU|nrefs:

In [6]:
# show BLEU & ds-BLEU scores
show_bleu_scores(mersenne_bleu, sys_fname_dict, max_n_order)

BLEU
MERSENNE-GEOS


,Llama,EuroLLM22,Qwen35-0.8B,Qwen35-9B
sent,51.0 (1.0),56.1 (1.0),30.5 (1.0),49.2 (1.0)
512,52.1 (1.0),57.2 (1.0),33.2 (1.0),49.0 (1.0)
1024,51.9 (1.0),56.9 (1.0),34.8 (0.93),49.5 (0.99)
1536,51.8 (1.0),56.7 (1.0),33.8 (0.9),50.3 (0.99)
2048,51.8 (1.0),57.2 (1.0),31.2 (0.85),51.3 (0.99)
3072,51.3 (0.99),56.8 (1.0),31.2 (0.81),52.5 (0.99)
4096,50.5 (0.98),55.2 (1.0),26.3 (0.69),52.9 (0.99)
doc,49.8 (0.98),56.1 (1.0),22.3 (0.59),54.4 (0.98)


\begin{tabular}{lllll}
 & Llama & EuroLLM22 & Qwen35-0.8B & Qwen35-9B \\
sent & 51.0 (1.0) & 56.1 (1.0) & 30.5 (1.0) & 49.2 (1.0) \\
512 & 52.1 (1.0) & 57.2 (1.0) & 33.2 (1.0) & 49.0 (1.0) \\
1024 & 51.9 (1.0) & 56.9 (1.0) & 34.8 (0.93) & 49.5 (0.99) \\
1536 & 51.8 (1.0) & 56.7 (1.0) & 33.8 (0.9) & 50.3 (0.99) \\
2048 & 51.8 (1.0) & 57.2 (1.0) & 31.2 (0.85) & 51.3 (0.99) \\
3072 & 51.3 (0.99) & 56.8 (1.0) & 31.2 (0.81) & 52.5 (0.99) \\
4096 & 50.5 (0.98) & 55.2 (1.0) & 26.3 (0.69) & 52.9 (0.99) \\
doc & 49.8 (0.98) & 56.1 (1.0) & 22.3 (0.59) & 54.4 (0.98) \\
\end{tabular}

ds-BLEU
MERSENNE-GEOS


,Llama,EuroLLM22,Qwen35-0.8B,Qwen35-9B
sent,52.3 \footnotesize{(1.0)},56.8 \footnotesize{(1.0)},35.5 \footnotesize{(0.99)},51.2 \footnotesize{(1.0)}
512,53.4 \footnotesize{(1.0)},58.0 \footnotesize{(1.0)},36.6 \footnotesize{(0.97)},50.9 \footnotesize{(0.99)}
1024,53.2 \footnotesize{(0.99)},57.8 \footnotesize{(1.0)},38.3 \footnotesize{(0.94)},51.5 \footnotesize{(0.99)}
1536,53.1 \footnotesize{(0.99)},57.5 \footnotesize{(1.0)},37.1 \footnotesize{(0.9)},52.0 \footnotesize{(0.99)}
2048,53.2 \footnotesize{(0.99)},57.9 \footnotesize{(0.99)},34.0 \footnotesize{(0.83)},53.0 \footnotesize{(0.99)}
3072,52.8 \footnotesize{(0.99)},57.7 \footnotesize{(1.0)},34.6 \footnotesize{(0.81)},54.0 \footnotesize{(0.99)}
4096,52.1 \footnotesize{(0.98)},56.7 \footnotesize{(1.0)},29.5 \footnotesize{(0.7)},54.3 \footnotesize{(0.99)}
doc,51.6 \footnotesize{(0.98)},57.1 \footnotesize{(0.99)},25.8 \footnotesize{(0.61)},55.4 \footnotesize{(0.98)}


\begin{tabular}{lllll}
 & Llama & EuroLLM22 & Qwen35-0.8B & Qwen35-9B \\
sent & 52.3 \footnotesize{(1.0)} & 56.8 \footnotesize{(1.0)} & 35.5 \footnotesize{(0.99)} & 51.2 \footnotesize{(1.0)} \\
512 & 53.4 \footnotesize{(1.0)} & 58.0 \footnotesize{(1.0)} & 36.6 \footnotesize{(0.97)} & 50.9 \footnotesize{(0.99)} \\
1024 & 53.2 \footnotesize{(0.99)} & 57.8 \footnotesize{(1.0)} & 38.3 \footnotesize{(0.94)} & 51.5 \footnotesize{(0.99)} \\
1536 & 53.1 \footnotesize{(0.99)} & 57.5 \footnotesize{(1.0)} & 37.1 \footnotesize{(0.9)} & 52.0 \footnotesize{(0.99)} \\
2048 & 53.2 \footnotesize{(0.99)} & 57.9 \footnotesize{(0.99)} & 34.0 \footnotesize{(0.83)} & 53.0 \footnotesize{(0.99)} \\
3072 & 52.8 \footnotesize{(0.99)} & 57.7 \footnotesize{(1.0)} & 34.6 \footnotesize{(0.81)} & 54.0 \footnotesize{(0.99)} \\
4096 & 52.1 \footnotesize{(0.98)} & 56.7 \footnotesize{(1.0)} & 29.5 \footnotesize{(0.7)} & 54.3 \footnotesize{(0.99)} \\
doc & 51.6 \footnotesize{(0.98)} & 57.1 \footnotesize{(0.99)} & 25.8 \f

In [20]:
table_ls = []
res_dict = mersenne_bleu.get_ds_bleu_corpus_dict(sys_fname_dict, show_bp = True, small_bp_size = True)
for subset in mersenne_bleu.get_sub_corpus_list():
    print(subset)
    ds_bleu_df = pd.DataFrame(res_dict[subset]).loc[max_n_order]
    ds_bleu_df.columns = pd.MultiIndex.from_product([[subset], ds_bleu_df.columns])
    table_ls.append(ds_bleu_df)

mersenne_table_df = pd.concat(table_ls, axis = 1)
display(mersenne_table_df)
print(mersenne_table_df.style.to_latex(column_format="l" + "r" * len(mersenne_table_df.columns)))

MERSENNE-GEOS


MERSENNE-GEOS                              \
                           Llama                   EuroLLM22   
sent   52.3 \footnotesize{(1.0)}   56.8 \footnotesize{(1.0)}   
512    53.4 \footnotesize{(1.0)}   58.0 \footnotesize{(1.0)}   
1024  53.2 \footnotesize{(0.99)}   57.8 \footnotesize{(1.0)}   
1536  53.1 \footnotesize{(0.99)}   57.5 \footnotesize{(1.0)}   
2048  53.2 \footnotesize{(0.99)}  57.9 \footnotesize{(0.99)}   
3072  52.8 \footnotesize{(0.99)}   57.7 \footnotesize{(1.0)}   
4096  52.1 \footnotesize{(0.98)}   56.7 \footnotesize{(1.0)}   
doc   51.6 \footnotesize{(0.98)}  57.1 \footnotesize{(0.99)}   

                                  
                       Qwen35-9B  
sent   51.2 \footnotesize{(1.0)}  
512   50.9 \footnotesize{(0.99)}  
1024  51.5 \footnotesize{(0.99)}  
1536  52.0 \footnotesize{(0.99)}  
2048  53.0 \footnotesize{(0.99)}  
3072  54.0 \footnotesize{(0.99)}  
4096  54.3 \footnotesize{(0.99)}  
doc   55.4 \footnotesize{(0.98)}

\begin{tabular}{lrrr}
 & \multicolumn{3}{r}{MERSENNE-GEOS} \\
 & Llama & EuroLLM22 & Qwen35-9B \\
sent & 52.3 \footnotesize{(1.0)} & 56.8 \footnotesize{(1.0)} & 51.2 \footnotesize{(1.0)} \\
512 & 53.4 \footnotesize{(1.0)} & 58.0 \footnotesize{(1.0)} & 50.9 \footnotesize{(0.99)} \\
1024 & 53.2 \footnotesize{(0.99)} & 57.8 \footnotesize{(1.0)} & 51.5 \footnotesize{(0.99)} \\
1536 & 53.1 \footnotesize{(0.99)} & 57.5 \footnotesize{(1.0)} & 52.0 \footnotesize{(0.99)} \\
2048 & 53.2 \footnotesize{(0.99)} & 57.9 \footnotesize{(0.99)} & 53.0 \footnotesize{(0.99)} \\
3072 & 52.8 \footnotesize{(0.99)} & 57.7 \footnotesize{(1.0)} & 54.0 \footnotesize{(0.99)} \\
4096 & 52.1 \footnotesize{(0.98)} & 56.7 \footnotesize{(1.0)} & 54.3 \footnotesize{(0.99)} \\
doc & 51.6 \footnotesize{(0.98)} & 57.1 \footnotesize{(0.99)} & 55.4 \footnotesize{(0.98)} \\
\end{tabular}



In [21]:
table_df = pd.concat([mersenne_table_df, student_table_df], axis = 1)

print(table_df.style.to_latex(column_format="l" + "r" * len(table_df.columns)))

\begin{tabular}{lrrrrrr}
 & \multicolumn{3}{r}{MERSENNE-GEOS} & \multicolumn{3}{r}{STUDENT} \\
 & Llama & EuroLLM22 & Qwen35-9B & Llama & EuroLLM22 & Qwen35-9B \\
sent & 52.3 \footnotesize{(1.0)} & 56.8 \footnotesize{(1.0)} & 51.2 \footnotesize{(1.0)} & 54.8 \footnotesize{(0.98)} & 62.3 \footnotesize{(0.98)} & 54.8 \footnotesize{(0.98)} \\
512 & 53.4 \footnotesize{(1.0)} & 58.0 \footnotesize{(1.0)} & 50.9 \footnotesize{(0.99)} & 55.6 \footnotesize{(0.97)} & 61.4 \footnotesize{(0.97)} & 51.9 \footnotesize{(0.95)} \\
1024 & 53.2 \footnotesize{(0.99)} & 57.8 \footnotesize{(1.0)} & 51.5 \footnotesize{(0.99)} & 54.4 \footnotesize{(0.96)} & 61.0 \footnotesize{(0.96)} & 51.6 \footnotesize{(0.93)} \\
1536 & 53.1 \footnotesize{(0.99)} & 57.5 \footnotesize{(1.0)} & 52.0 \footnotesize{(0.99)} & 53.7 \footnotesize{(0.95)} & 60.9 \footnotesize{(0.97)} & 51.5 \footnotesize{(0.91)} \\
2048 & 53.2 \footnotesize{(0.99)} & 57.9 \footnotesize{(0.99)} & 53.0 \footnotesize{(0.99)} & 51.2 \footnotesize{(0.9

### 2. ds-BLEU with t-test

In [28]:
to_test_max_n = ['sent'] + sorted(list(valid_max_n)) + [ 'doc', 'sent', 'doc' ]


show_ttest_ds_bleu(mersenne_bleu, to_test_max_n, sys_fname_dict )

Llama
EuroLLM22
Qwen35-9B


,Llama,EuroLLM22,Qwen35-9B
sent-512,-1.1,-1.2,-
512-1024,-,-,\mark{-0.6}
1024-1536,-,-,\mark{-0.6}
1536-2048,-,-,-1.0
2048-3072,\mark{0.4},-,\mark{-0.9}
3072-4096,-,-,-
4096-doc,-,-,-1.1
doc-sent,-,-,4.2
sent-doc,-,-,-4.2


\begin{tabular}{llll}
 & Llama & EuroLLM22 & Qwen35-9B \\
sent-512 & -1.1 & -1.2 & - \\
512-1024 & - & - & \mark{-0.6} \\
1024-1536 & - & - & \mark{-0.6} \\
1536-2048 & - & - & -1.0 \\
2048-3072 & \mark{0.4} & - & \mark{-0.9} \\
3072-4096 & - & - & - \\
4096-doc & - & - & -1.1 \\
doc-sent & - & - & 4.2 \\
sent-doc & - & - & -4.2 \\
\end{tabular}

Llama
EuroLLM22
Qwen35-9B


,Llama,EuroLLM22,Qwen35-9B
sent-512,-1.1 \footnotesize{(0.0)},-1.2 \footnotesize{(0.0)},0.4 \footnotesize{(0.22)}
512-1024,0.1 \footnotesize{(0.39)},0.3 \footnotesize{(0.12)},\mark{-0.6 \footnotesize{(0.01)}}
1024-1536,0.1 \footnotesize{(0.59)},0.3 \footnotesize{(0.06)},\mark{-0.6 \footnotesize{(0.05)}}
1536-2048,-0.1 \footnotesize{(0.38)},-0.4 \footnotesize{(0.05)},-1.0 \footnotesize{(0.01)}
2048-3072,\mark{0.4 \footnotesize{(0.04)}},0.2 \footnotesize{(0.23)},\mark{-0.9 \footnotesize{(0.01)}}
3072-4096,0.7 \footnotesize{(0.11)},0.9 \footnotesize{(0.3)},-0.4 \footnotesize{(0.08)}
4096-doc,0.5 \footnotesize{(0.47)},-0.4 \footnotesize{(0.65)},-1.1 \footnotesize{(0.01)}
doc-sent,-0.7 \footnotesize{(0.34)},0.4 \footnotesize{(0.41)},4.2 \footnotesize{(0.0)}
sent-doc,0.7 \footnotesize{(0.34)},-0.4 \footnotesize{(0.41)},-4.2 \footnotesize{(0.0)}


### 3. COMET

In [38]:


mersenneComet = CometEvaluator(
    dataset_paths = mersenne_data_path,
    system_paths = mersenne_mt_path,
    valid_max_n = valid_max_n,
    corpus_name = corpus_name,
    sub_corpus_list = None,
    sep_tag = '<sep>',
    )

assert len(mersenneComet.get_sub_corpus_list()) == 1
subset = corpus_name


In [30]:
# gather comet scores, compute ds-comet
for model_type in model_list:  
    for subset in mersenneComet.get_sub_corpus_list():
        mersenneComet.store_comet_score(model_type, subset=subset, is_json = False)
        mersenneComet.get_ds_comet_and_store(model_type, subset=subset)


In [41]:
# show COMET
comet_sent_dict = mersenneComet.get_comet_sent_dict(sys_fname_dict)

order = ['sent'] + [str(v) for v in sorted(valid_max_n)] + ['doc']
comet_df = pd.concat([ v.rename(columns = {subset: k}) for k, v in comet_sent_dict.items() ], axis = 1).loc[order]


# print(comet_df.style.format(precision=2).to_latex())

mersenne_comet_df = comet_df*100
mersenne_comet_df.columns = pd.MultiIndex.from_product([[subset], mersenne_comet_df.columns])

mersenne_comet_df

MERSENNE-GEOS                      
             Llama  EuroLLM22  Qwen35-9B
sent     88.590072  89.822360  89.341960
512      88.414652  89.770931  88.984665
1024     88.020034  89.691145  88.973461
1536     88.180763  89.555996  88.988142
2048     88.076372  89.540453  89.163091
3072     87.735349  89.529251  89.329815
4096     87.074600  88.810306  89.344712
doc      86.427971  89.213052  89.185555

In [45]:
res_df = pd.concat([ mersenne_comet_df, student_comet_df], axis = 1)

display(res_df)
print(res_df.style.format(precision = 1).to_latex(column_format="l" + "r" * len(res_df.columns)))

MERSENNE-GEOS                          STUDENT                      
             Llama  EuroLLM22  Qwen35-9B      Llama  EuroLLM22  Qwen35-9B
sent     88.590072  89.822360  89.341960  83.948756  85.965305  85.495267
512      88.414652  89.770931  88.984665  83.151425  85.067541  83.862023
1024     88.020034  89.691145  88.973461  82.600567  84.555943  83.118286
1536     88.180763  89.555996  88.988142  81.346566  84.648394  82.018297
2048     88.076372  89.540453  89.163091  80.327004  81.855257  82.518110
3072     87.735349  89.529251  89.329815  78.500209  83.856370  82.669234
4096     87.074600  88.810306  89.344712  77.895796  81.682358  82.445143
doc      86.427971  89.213052  89.185555  66.368625  75.970782  81.995448

\begin{tabular}{lrrrrrr}
 & \multicolumn{3}{r}{MERSENNE-GEOS} & \multicolumn{3}{r}{STUDENT} \\
 & Llama & EuroLLM22 & Qwen35-9B & Llama & EuroLLM22 & Qwen35-9B \\
sent & 88.6 & 89.8 & 89.3 & 83.9 & 86.0 & 85.5 \\
512 & 88.4 & 89.8 & 89.0 & 83.2 & 85.1 & 83.9 \\
1024 & 88.0 & 89.7 & 89.0 & 82.6 & 84.6 & 83.1 \\
1536 & 88.2 & 89.6 & 89.0 & 81.3 & 84.6 & 82.0 \\
2048 & 88.1 & 89.5 & 89.2 & 80.3 & 81.9 & 82.5 \\
3072 & 87.7 & 89.5 & 89.3 & 78.5 & 83.9 & 82.7 \\
4096 & 87.1 & 88.8 & 89.3 & 77.9 & 81.7 & 82.4 \\
doc & 86.4 & 89.2 & 89.2 & 66.4 & 76.0 & 82.0 \\
\end{tabular}



In [32]:
to_test_max_n = ['sent'] + sorted(list(valid_max_n)) + [ 'doc', 'sent', 'doc' ]


show_ttest_comet(mersenneComet, to_test_max_n, sys_fname_dict )

Llama
sent-512 0.2 TtestResult(statistic=np.float64(2.093240581242386), pvalue=np.float64(0.050757713269907724), df=np.int64(18))
512-1024 0.5 TtestResult(statistic=np.float64(2.055596772704), pvalue=np.float64(0.05462378335511411), df=np.int64(18))
1024-1536 -0.3 TtestResult(statistic=np.float64(-1.2824473275968964), pvalue=np.float64(0.2159560399450588), df=np.int64(18))
1536-2048 0.1 TtestResult(statistic=np.float64(0.9667783446811737), pvalue=np.float64(0.34646590946056244), df=np.int64(18))
2048-3072 0.5 TtestResult(statistic=np.float64(1.0505103904919537), pvalue=np.float64(0.3073836663206767), df=np.int64(18))
3072-4096 0.8 TtestResult(statistic=np.float64(1.4571384326905008), pvalue=np.float64(0.16230307806601946), df=np.int64(18))
4096-doc 0.3 TtestResult(statistic=np.float64(0.343474668881207), pvalue=np.float64(0.735220672069947), df=np.int64(18))
doc-sent -2.1 TtestResult(statistic=np.float64(-2.9869545808986993), pvalue=np.float64(0.007906463642960895), df=np.int64(18))
se

,Llama,EuroLLM22,Qwen35-9B
sent-512,-,-,0.4
512-1024,-,0.1,-
1024-1536,-,-,-
1536-2048,-,-,-
2048-3072,-,-,\mark{-0.2}
3072-4096,-,-,-
4096-doc,-,-,-
doc-sent,-2.1,-0.6,-
sent-doc,2.1,0.6,-


\begin{tabular}{llll}
 & Llama & EuroLLM22 & Qwen35-9B \\
sent-512 & - & - & 0.4 \\
512-1024 & - & 0.1 & - \\
1024-1536 & - & - & - \\
1536-2048 & - & - & - \\
2048-3072 & - & - & \mark{-0.2} \\
3072-4096 & - & - & - \\
4096-doc & - & - & - \\
doc-sent & -2.1 & -0.6 & - \\
sent-doc & 2.1 & 0.6 & - \\
\end{tabular}

Llama
sent-512 0.2 TtestResult(statistic=np.float64(2.093240581242386), pvalue=np.float64(0.050757713269907724), df=np.int64(18))
512-1024 0.5 TtestResult(statistic=np.float64(2.055596772704), pvalue=np.float64(0.05462378335511411), df=np.int64(18))
1024-1536 -0.3 TtestResult(statistic=np.float64(-1.2824473275968964), pvalue=np.float64(0.2159560399450588), df=np.int64(18))
1536-2048 0.1 TtestResult(statistic=np.float64(0.9667783446811737), pvalue=np.float64(0.34646590946056244), df=np.int64(18))
2048-3072 0.5 TtestResult(statistic=np.float64(1.0505103904919537), pvalue=np.float64(0.3073836663206767), df=np.int64(18))
3072-4096 0.8 TtestResult(statistic=np.float64(1.457138432

,Llama,EuroLLM22,Qwen35-9B
sent-512,-,-,0.4 \footnotesize{(0.0)}
512-1024,-,0.1 \footnotesize{(0.0)},-
1024-1536,-,-,-
1536-2048,-,-,-
2048-3072,-,-,-0.2 \footnotesize{(0.01)}
3072-4096,-,-,-
4096-doc,-,-,-
doc-sent,-2.1 \footnotesize{(0.01)},-0.6 \footnotesize{(0.01)},-
sent-doc,2.1 \footnotesize{(0.01)},0.6 \footnotesize{(0.01)},-


### 4. Position bias -- Corr(begin_id, comet) 

In [33]:
mersennePos = PositionBiasAnalyzer(
    dataset_paths = mersenne_data_path,
    system_paths = mersenne_mt_path,
    valid_max_n = valid_max_n,
    corpus_name = corpus_name,
    sub_corpus_list = None,
    )

assert len(mersennePos.get_sub_corpus_list()) == 1
subset = corpus_name


In [34]:
disable_full_doc = False
nb_positions = len(valid_max_n)+1
total = get_nb_sample_for_position_bias(mersennePos, sys_fname_dict, nb_positions, disable_full_doc = disable_full_doc)
print(total)
compute_position_bias(mersennePos, sys_fname_dict, nb_positions, disable_full_doc = disable_full_doc, show_p_val = False)

set of different begin idx
MERSENNE-GEOS, 454, maximum different positions = 7
MERSENNE-GEOS, 454, maximum different positions = 7
MERSENNE-GEOS, 454, maximum different positions = 7
454
454
454
Comparing positon bias, make sure that models in sys_fname_dict share the same tokenizer
[159, 384, 674, 950, 1219, 2327, 6644]
Llama
454
454
Comparing positon bias, make sure that models in sys_fname_dict share the same tokenizer
[170, 409, 716, 1012, 1300, 2487, 7180]
EuroLLM22
454
454
Comparing positon bias, make sure that models in sys_fname_dict share the same tokenizer
[163, 392, 689, 973, 1249, 2389, 6871]
Qwen35-9B


,index,Llama,index,EuroLLM22,index,Qwen35-9B
$p$0-$p$1,159-384,-,170-409,-,163-392,\mark{-0.4}
$p$1-$p$2,384-674,\mark{-0.2},409-716,-,392-689,-
$p$2-$p$3,674-950,-,716-1012,-,689-973,-
$p$3-$p$4,950-1219,-,1012-1300,-,973-1249,-
$p$4-$p$5,1219-2327,0.8,1300-2487,-,1249-2389,-
$p$5-$p$6,2327-6644,1.7,2487-7180,-,2389-6871,-


\begin{tabular}{lllllll}
 & index & Llama & index & EuroLLM22 & index & Qwen35-9B \\
$p$0-$p$1 & 159-384 & - & 170-409 & - & 163-392 & \mark{-0.4} \\
$p$1-$p$2 & 384-674 & \mark{-0.2} & 409-716 & - & 392-689 & - \\
$p$2-$p$3 & 674-950 & - & 716-1012 & - & 689-973 & - \\
$p$3-$p$4 & 950-1219 & - & 1012-1300 & - & 973-1249 & - \\
$p$4-$p$5 & 1219-2327 & 0.8 & 1300-2487 & - & 1249-2389 & - \\
$p$5-$p$6 & 2327-6644 & 1.7 & 2487-7180 & - & 2389-6871 & - \\
\end{tabular}



,Llama,EuroLLM22,Qwen35-9B
$p$0-$p$1,-,-,\mark{-0.4}
$p$1-$p$2,\mark{-0.2},-,-
$p$2-$p$3,-,-,-
$p$3-$p$4,-,-,-
$p$4-$p$5,0.8,-,-
$p$5-$p$6,1.7,-,-


\begin{tabular}{llll}
 & Llama & EuroLLM22 & Qwen35-9B \\
$p$0-$p$1 & - & - & \mark{-0.4} \\
$p$1-$p$2 & \mark{-0.2} & - & - \\
$p$2-$p$3 & - & - & - \\
$p$3-$p$4 & - & - & - \\
$p$4-$p$5 & 0.8 & - & - \\
$p$5-$p$6 & 1.7 & - & - \\
\end{tabular}



In [35]:
disable_full_doc = True
nb_positions = len(valid_max_n)
total = get_nb_sample_for_position_bias(mersennePos, sys_fname_dict, nb_positions, disable_full_doc = disable_full_doc)
print(total)
compute_position_bias(mersennePos, sys_fname_dict, nb_positions, disable_full_doc = disable_full_doc, show_p_val = False)

set of different begin idx
MERSENNE-GEOS, 665, maximum different positions = 6
MERSENNE-GEOS, 665, maximum different positions = 6
MERSENNE-GEOS, 665, maximum different positions = 6
665
665
665
Comparing positon bias, make sure that models in sys_fname_dict share the same tokenizer
[163, 405, 628, 837, 1317, 2718]
Llama
665
665
Comparing positon bias, make sure that models in sys_fname_dict share the same tokenizer
[174, 431, 668, 892, 1410, 2918]
EuroLLM22
665
665
Comparing positon bias, make sure that models in sys_fname_dict share the same tokenizer
[166, 413, 641, 856, 1351, 2792]
Qwen35-9B


,index,Llama,index,EuroLLM22,index,Qwen35-9B
$p$0-$p$1,163-405,-,174-431,-,166-413,\mark{-0.3}
$p$1-$p$2,405-628,\mark{-0.2},431-668,-,413-641,-
$p$2-$p$3,628-837,-,668-892,-,641-856,-
$p$3-$p$4,837-1317,-,892-1410,-,856-1351,-
$p$4-$p$5,1317-2718,1.6,1410-2918,-,1351-2792,-


\begin{tabular}{lllllll}
 & index & Llama & index & EuroLLM22 & index & Qwen35-9B \\
$p$0-$p$1 & 163-405 & - & 174-431 & - & 166-413 & \mark{-0.3} \\
$p$1-$p$2 & 405-628 & \mark{-0.2} & 431-668 & - & 413-641 & - \\
$p$2-$p$3 & 628-837 & - & 668-892 & - & 641-856 & - \\
$p$3-$p$4 & 837-1317 & - & 892-1410 & - & 856-1351 & - \\
$p$4-$p$5 & 1317-2718 & 1.6 & 1410-2918 & - & 1351-2792 & - \\
\end{tabular}



,Llama,EuroLLM22,Qwen35-9B
$p$0-$p$1,-,-,\mark{-0.3}
$p$1-$p$2,\mark{-0.2},-,-
$p$2-$p$3,-,-,-
$p$3-$p$4,-,-,-
$p$4-$p$5,1.6,-,-


\begin{tabular}{llll}
 & Llama & EuroLLM22 & Qwen35-9B \\
$p$0-$p$1 & - & - & \mark{-0.3} \\
$p$1-$p$2 & \mark{-0.2} & - & - \\
$p$2-$p$3 & - & - & - \\
$p$3-$p$4 & - & - & - \\
$p$4-$p$5 & 1.6 & - & - \\
\end{tabular}

